# Annotation Validation — Coverage and Agreement against MEVA

This notebook validates the pipeline's event annotations against the MEVA ground truth in two directions, with
one setup, one judge and one set of credentials. It merges the earlier coverage notebook with the QA
`annotations-report` notebook, so the two checks are always run on the same data with the same judge.

| Part | Question | Unit | Direction | Output table |
|---|---|---|---|---|
| **A. Coverage** | Do the generated annotations describe the activities MEVA recorded? | one **(video, MEVA activity type)** pair | recall | `bronze.meva_event_coverage` |
| **B. Agreement** | Does each generated annotation agree with the MEVA activities happening in its time window? | one **generated event** | precision | `bronze.annotations_report` |

**Part A** ignores timing: an annotation covers an activity type if it describes it anywhere in the same video, and
one annotation is enough however many instances MEVA has. **Part B** keeps timing: each generated event is matched
to the MEVA activities whose frame ranges overlap its window (the matching stage that used to live outside this
repository and wrote `bronze.matched_pairs` is rebuilt here from `bronze.ground_truth`), and the judge scores how
well the annotation text agrees with those activities. A low coverage means the pipeline misses things; a low
agreement means it describes things that are not there.

**Workflow:**
1. Connect to the PostgreSQL `bronze` schema (env vars locally, Kaggle Secrets on Kaggle) and load one judge, local
   MLX on Apple Silicon or Groq elsewhere.
2. Part A: build the comparison set, judge every (video, activity type) pair, report coverage, write the table.
3. Part B: match every generated event to its overlapping MEVA activities, judge agreement with a reference-based
   rubric, report the match rate, write the table.
4. Put the two numbers side by side, overall and per video.

**Scope:** semantic checks only. Temporal IoU and spatial IoU are separate evaluation stages.


## 1. Environment and database connection

Install the libraries, then resolve credentials. The same code runs locally and on Kaggle: each secret is read
from an environment variable first (`.env`, loaded with python-dotenv) and from Kaggle Secrets as a fallback.

| Value | Env var (local `.env`) | Kaggle Secret |
|---|---|---|
| Database | `SUPABASE_DB_URL` — or `SUPABASE_DB_PASSWORD` (+ optional `SUPABASE_DB_USER` / `_HOST` / `_PORT` / `_NAME`) | `DB_PASSWORD` |
| LLM judge (Groq backend only) | `GROQ_API_KEY` | `groq` |
| Judge model (optional) | `GROQ_MODEL`, `MLX_JUDGE_MODEL` | — |
| Judge backend (optional) | `JUDGE_BACKEND` = `auto` / `mlx` / `groq` | — |

On macOS the judge runs locally and **no Groq key is needed**. See `.env.example` in the repository root.

In [1]:
import platform
import subprocess
import sys

# mlx-lm only exists on macOS; groq is installed everywhere so either backend can be forced.
packages = ["pandas", "sqlalchemy", "psycopg2-binary", "python-dotenv", "groq"]
if platform.system() == "Darwin":
    packages.append("mlx-lm")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Installed:", ", ".join(packages))

Installed: pandas, sqlalchemy, psycopg2-binary, python-dotenv, groq, mlx-lm


In [2]:
import json
import os
import platform
import re
from datetime import datetime, timezone

import pandas as pd
from sqlalchemy import URL, create_engine, text

try:  # local runs
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass

try:  # Kaggle runs
    from kaggle_secrets import UserSecretsClient

    _kaggle_secrets = UserSecretsClient()
except Exception:
    _kaggle_secrets = None


def get_secret(env_var, kaggle_name=None, default=None, required=True):
    """Read a credential from the environment, then from Kaggle Secrets."""
    value = os.environ.get(env_var)

    if not value and _kaggle_secrets is not None and kaggle_name:
        try:
            value = _kaggle_secrets.get_secret(kaggle_name)
        except Exception:
            value = None

    if not value:
        value = default

    if not value and required:
        raise RuntimeError(
            f"Missing credential: set the {env_var} environment variable"
            + (f" or the Kaggle Secret '{kaggle_name}'." if kaggle_name else ".")
        )

    return value


print("Kaggle Secrets available:", _kaggle_secrets is not None)

Kaggle Secrets available: False


In [3]:
DATABASE_URL = os.environ.get("SUPABASE_DB_URL")

if not DATABASE_URL:
    DATABASE_URL = URL.create(
        drivername="postgresql+psycopg2",
        username=get_secret("SUPABASE_DB_USER", default="postgres.uqofqjychtjsvfioebod"),
        password=get_secret("SUPABASE_DB_PASSWORD", kaggle_name="DB_PASSWORD"),
        host=get_secret("SUPABASE_DB_HOST", default="aws-0-ap-south-1.pooler.supabase.com"),
        port=int(get_secret("SUPABASE_DB_PORT", default="5432")),
        database=get_secret("SUPABASE_DB_NAME", default="postgres"),
    )
elif DATABASE_URL.startswith("postgresql://"):
    DATABASE_URL = DATABASE_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print("SQLAlchemy engine created.")

SQLAlchemy engine created.


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user;")).fetchone()

print("Connection successful.")
print("Database:", result[0])
print("User:", result[1])

Connection successful.
Database: postgres
User: postgres


## 2. Judge settings

The judge backend is chosen from the operating system: **Apple Silicon macOS runs the judge locally with MLX**
(`mlx-community/Qwen3-0.6B-8bit`, the same local-model approach notebook 05 uses for annotation), and any other
platform — Kaggle included — calls Groq. Set `JUDGE_BACKEND` to `mlx` or `groq` to override the detection.

The two backends are asked in different shapes. Groq judges a whole video in one call: the annotation list once,
then every MEVA activity type of that video. The local 0.6B model cannot answer about several activities at once
without copying its own output template, so it is asked one activity at a time.

In [5]:
COVERAGE_THRESHOLD = 0.7   # judge score (0.0-1.0 in steps of 0.1) from which a MEVA activity counts as covered
COVERAGE_TABLE = "meva_event_coverage"
WRITE_TO_DB = True         # set to False to keep the report local
# None: judge every pair and rebuild the whole table. A substring of video_name (a full name for one
# video, a camera id such as "G421" for a camera): judge only those videos and replace only their rows,
# leaving the rest of the table as it is. Used to test a prompt change on a pilot video.
VIDEO_FILTER = None  # Pilot video. Set to None to judge everything.

# Part B (agreement) settings. The rubric is tagged with a prompt version so earlier versions written by the
# QA notebook (v1_explicit_judge, v2.3_semantic_judge) stay in the table for comparison; bump it when the rubric
# changes. An event matches when its score reaches AGREEMENT_THRESHOLD.
AGREEMENT_TABLE = "annotations_report"
MATCHED_PAIRS_TABLE = "matched_pairs"   # the event-to-MEVA matches behind Part B, kept for the QA notebook
AGREEMENT_PROMPT_VERSION = "v4_unified_scale_judge"
AGREEMENT_THRESHOLD = 0.7   # same pass mark as coverage: scores of 0.7 and up match
AGREEMENT_BATCH_SIZE = 20   # events per hosted call; the local judge always takes one event per call
FRAME_TOLERANCE = 0         # extra frames either side of an event window when matching MEVA activities

GROQ_JUDGE_MODEL = get_secret("GROQ_MODEL", default="openai/gpt-oss-120b", required=False)
# Set MLX_JUDGE_MODEL to override. Smaller local models were tried first and were not usable:
# Qwen3-0.6B-8bit marked every activity in a video as covered (see the warning it still prints).
MLX_JUDGE_MODEL = get_secret(
    "MLX_JUDGE_MODEL", default="mlx-community/Qwen3-8B-4bit", required=False
)


def resolve_judge_backend():
    """Local MLX on Apple Silicon macOS, Groq anywhere else. JUDGE_BACKEND overrides."""
    requested = os.environ.get("JUDGE_BACKEND", "auto").strip().lower()

    if requested in {"mlx", "groq"}:
        return requested

    if requested not in {"", "auto"}:
        raise ValueError(f"JUDGE_BACKEND must be auto, mlx or groq - got {requested!r}")

    is_apple_silicon = platform.system() == "Darwin" and platform.machine() == "arm64"
    return "mlx" if is_apple_silicon else "groq"


JUDGE_BACKEND = resolve_judge_backend()
JUDGE_MODEL = MLX_JUDGE_MODEL if JUDGE_BACKEND == "mlx" else GROQ_JUDGE_MODEL

print("Operating system: ", platform.system(), platform.machine())
print("Judge backend:    ", JUDGE_BACKEND, "(local)" if JUDGE_BACKEND == "mlx" else "(hosted)")
print("Judge model:      ", JUDGE_MODEL)
print("Asked per call:   ", "one activity" if JUDGE_BACKEND == "mlx" else "one whole video")
print("Coverage cutoff:  ", COVERAGE_THRESHOLD)
print("Coverage table:   ", f"bronze.{COVERAGE_TABLE}")
print("Agreement table:  ", f"bronze.{AGREEMENT_TABLE} (prompt {AGREEMENT_PROMPT_VERSION}, match at >= {AGREEMENT_THRESHOLD})")
print("Video filter:     ", VIDEO_FILTER or "none (all videos)")

if JUDGE_BACKEND == "mlx" and re.search(r"-(0\.6B|1\.7B|4B)-", JUDGE_MODEL):
    print(
        "\nWARNING: this local model is smaller than the tested default. Qwen3-0.6B-8bit, measured\n"
        "on camera G340's 13:15 video (15 MEVA activity types, 11 annotations), marked all 15 as\n"
        "covered and credited one annotation - 'White SUV enters from the left' - for most of them,\n"
        "scoring almost everything 0.75 with a canned reason. Splitting it into one question per\n"
        "annotation did not help. Check section 10.2 before reporting anything from a small judge."
    )

elif JUDGE_BACKEND == "mlx":
    print(
        "\nNote: a local judge is still weaker than the hosted 120B model. Check section 10.2\n"
        "before reporting these numbers, and re-run with JUDGE_BACKEND=groq to compare."
    )

Operating system:  Darwin arm64
Judge backend:     mlx (local)
Judge model:       mlx-community/Qwen3-8B-4bit
Asked per call:    one activity
Coverage cutoff:   0.7
Coverage table:    bronze.meva_event_coverage
Agreement table:   bronze.annotations_report (prompt v4_unified_scale_judge, match at >= 0.7)
Video filter:      none (all videos)

Note: a local judge is still weaker than the hosted 120B model. Check section 10.2
before reporting these numbers, and re-run with JUDGE_BACKEND=groq to compare.


# Part A — Coverage: do the annotations describe what MEVA recorded?

## 3. Build the comparison set

One row per (processed video, MEVA activity type). Only videos present in `bronze.videos` take part: MEVA has
ground truth for hundreds of videos the pipeline never ingested, and those are an ingestion gap rather than a
description failure, so they are left out entirely. `bronze.ground_truth` is collapsed to the distinct activity types of
each video — the instance count is kept as `n_meva_instances`, and the ground-truth ids as `gt_ids`, but they
play no part in the comparison. Every generated event of that video becomes a candidate annotation. No frame
condition is used anywhere.

Rows are tagged with a `status`:

* `judged` — the video has generated annotations, so the pair goes to the judge.
* `no_annotations` — the pipeline produced nothing for this video. Not covered by definition; no LLM call needed.


In [6]:
# bronze.events gains `actions` (checklist phrases the annotator named) from notebook 07 onwards;
# older loads do not have the column, so it is only selected when present.
EVENTS_HAVE_ACTIONS = bool(
    pd.read_sql(
        text("""
        SELECT EXISTS (
            SELECT 1 FROM information_schema.columns
            WHERE table_schema = 'bronze' AND table_name = 'events' AND column_name = 'actions'
        ) AS has_actions;
        """),
        engine,
    ).iloc[0, 0]
)
ACTIONS_SQL = ",\n                'actions', COALESCE(e.actions, '[]'::jsonb)" if EVENTS_HAVE_ACTIONS else ""
print("bronze.events has an actions column:", EVENTS_HAVE_ACTIONS)

comparison_sql = text(f"""
WITH meva AS (
    SELECT
        g.video_id,
        g.camera_id,
        g.capture_start_local,
        g.activity_type,
        COUNT(*) AS n_meva_instances,
        array_agg(g.id ORDER BY g.id) AS gt_ids,
        MAX(v.video_name) AS video_name
    FROM bronze.ground_truth g
    JOIN bronze.videos v              -- only videos the pipeline processed and loaded
        ON v.video_id = g.video_id
    GROUP BY g.video_id, g.camera_id, g.capture_start_local, g.activity_type
),
annotations AS (
    SELECT
        e.video_id,
        COUNT(*) AS n_annotations,
        json_agg(
            json_build_object(
                'event_id', e.event_id,
                'event_name', e.event_name,
                'description', e.description{ACTIONS_SQL}
            )
            ORDER BY e.start_frame
        ) AS annotations
    FROM bronze.events e
    GROUP BY e.video_id
)
SELECT
    meva.video_id || ' | ' || meva.activity_type AS pair_id,
    meva.camera_id,
    meva.capture_start_local,
    meva.video_id,
    meva.video_name,
    meva.activity_type,
    meva.n_meva_instances,
    meva.gt_ids,
    COALESCE(annotations.n_annotations, 0) AS n_annotations,
    COALESCE(annotations.annotations, '[]') AS annotations
FROM meva
LEFT JOIN annotations ON annotations.video_id = meva.video_id
ORDER BY meva.camera_id, meva.capture_start_local, meva.activity_type;
""")

comparison_set = pd.read_sql(comparison_sql, engine)

comparison_set["annotations"] = comparison_set["annotations"].apply(
    lambda value: value if isinstance(value, list) else json.loads(value)
)

comparison_set["status"] = "judged"
comparison_set.loc[comparison_set["n_annotations"] == 0, "status"] = "no_annotations"

print("Processed videos with MEVA ground truth:", comparison_set["video_id"].nunique())
print("MEVA activity types to compare:", len(comparison_set))
print("MEVA instances behind them:   ", int(comparison_set["n_meva_instances"].sum()))
print()
print(comparison_set["status"].value_counts().to_string())

comparison_set[["video_name", "camera_id", "capture_start_local", "activity_type",
                "n_meva_instances", "n_annotations", "status"]].head(12)

bronze.events has an actions column: True
Processed videos with MEVA ground truth: 70
MEVA activity types to compare: 605
MEVA instances behind them:    2720

status
judged    605


,video_name,camera_id,capture_start_local,activity_type,n_meva_instances,n_annotations,status
0,2018-03-07.11-10-00.11-15-00.school.G299,G299,2018-03-07 11:10:00,person_talks_on_phone,1,7,judged
1,2018-03-07.11-10-00.11-15-00.school.G299,G299,2018-03-07 11:10:00,person_texts_on_phone,3,7,judged
2,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_carries_heavy_object,1,16,judged
3,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_enters_scene_through_structure,3,16,judged
4,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_exits_scene_through_structure,4,16,judged
5,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_talks_on_phone,1,16,judged
6,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_talks_to_person,1,16,judged
7,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,person_texts_on_phone,1,16,judged
8,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,vehicle_reverses,1,16,judged
9,2018-03-07.11-00-06.11-05-06.school.G300,G300,2018-03-07 11:00:06,vehicle_starts,1,16,judged


### 3.1 Which objects each MEVA activity implies

MEVA ground truth stores an activity type and an actor count, not object labels, so the expected objects are
derived from the activity name itself: `person_opens_trunk` implies a person and a vehicle. The check then looks
for those object words in the generated annotation text.

The check runs **per annotation**, not over the video's pooled text. Pooling was useless: every video that has
annotations mentions both a person and some vehicle word somewhere, so nothing was ever reported missing.
Per annotation it is informative. `has_annotation_with_all_objects` is the yes/no answer — does at least one
of the video's annotations name every object the activity implies — and `n_annotations_with_all_objects` keeps
the count behind it. After judging, `covering_objects_missing` says whether the one annotation credited with
covering the activity actually named them.

In [7]:
OBJECT_KEYWORDS = {
    "person": ("person", "people", "man", "woman", "pedestrian", "individual", "someone"),
    "vehicle": ("vehicle", "car", "suv", "truck", "van", "sedan", "bus", "motorcycle", "hatchback"),
}


def meva_required_objects(activity_type):
    """Actor classes implied by a MEVA activity name."""
    activity = activity_type.lower()
    required = []

    if "person" in activity or "hand" in activity:
        required.append("person")
    if "vehicle" in activity or "trunk" in activity:
        required.append("vehicle")

    return required


def objects_missing_from_annotation(required, annotation):
    """Required object classes that this one annotation does not mention by name."""
    text = f"{annotation.get('event_name', '')} {annotation.get('description', '')}".lower()

    return [
        obj for obj in required
        if not any(keyword in text for keyword in OBJECT_KEYWORDS[obj])
    ]


def count_annotations_with_all_objects(required, annotations):
    """How many of the video's annotations name every object the activity implies."""
    return sum(
        1 for annotation in annotations
        if not objects_missing_from_annotation(required, annotation)
    )


comparison_set["required_objects"] = comparison_set["activity_type"].apply(meva_required_objects)
comparison_set["n_annotations_with_all_objects"] = comparison_set.apply(
    lambda row: count_annotations_with_all_objects(row["required_objects"], row["annotations"]),
    axis=1,
)
comparison_set["has_annotation_with_all_objects"] = comparison_set["n_annotations_with_all_objects"] > 0

comparison_set[["camera_id", "activity_type", "required_objects", "n_annotations",
                "has_annotation_with_all_objects", "n_annotations_with_all_objects",
                "status"]].head(12)

,camera_id,activity_type,required_objects,n_annotations,has_annotation_with_all_objects,n_annotations_with_all_objects,status
0,G299,person_talks_on_phone,[person],7,True,7,judged
1,G299,person_texts_on_phone,[person],7,True,7,judged
2,G300,person_carries_heavy_object,[person],16,True,11,judged
3,G300,person_enters_scene_through_structure,[person],16,True,11,judged
4,G300,person_exits_scene_through_structure,[person],16,True,11,judged
5,G300,person_talks_on_phone,[person],16,True,11,judged
6,G300,person_talks_to_person,[person],16,True,11,judged
7,G300,person_texts_on_phone,[person],16,True,11,judged
8,G300,vehicle_reverses,[vehicle],16,True,10,judged
9,G300,vehicle_starts,[vehicle],16,True,10,judged


## 4. Pairs to evaluate

Every pair in the comparison set is judged on every run, and section 11 rebuilds `bronze.meva_event_coverage`
from the result. Nothing is skipped as "already evaluated": re-annotated videos, new ground truth and a switch
of judge model are all picked up simply by running the notebook again. The cost is one full pass of judge
calls per run.

With `VIDEO_FILTER` set (section 2) only the matching videos are judged, and section 11 replaces just their
rows, so a prompt change can be tested on one pilot video without re-judging or wiping the others.

The earlier frame-based design wrote `bronze.meva_coverage_report`, which has a different shape (one row per
ground-truth instance, with frame columns and a frame tolerance); that table is left as it is.


In [8]:
report_exists = pd.read_sql(
    text("""
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = 'bronze'
          AND table_name = :table_name
    ) AS report_exists;
    """),
    engine,
    params={"table_name": COVERAGE_TABLE},
).iloc[0, 0]

existing_rows = 0
if report_exists:
    if VIDEO_FILTER:
        existing_rows = int(pd.read_sql(
            text(f"SELECT COUNT(*) FROM bronze.{COVERAGE_TABLE} WHERE video_name LIKE :pattern;"),
            engine, params={"pattern": f"%{VIDEO_FILTER}%"},
        ).iloc[0, 0])
    else:
        existing_rows = int(
            pd.read_sql(text(f"SELECT COUNT(*) FROM bronze.{COVERAGE_TABLE};"), engine).iloc[0, 0]
        )

# Every selected pair is evaluated on every run; section 11 replaces its rows.
if VIDEO_FILTER:
    pending = comparison_set[
        comparison_set["video_name"].str.contains(VIDEO_FILTER, regex=False, na=False)
    ].copy()
    if pending.empty:
        raise ValueError(f"No MEVA pairs match VIDEO_FILTER={VIDEO_FILTER!r}; check the name in bronze.videos.")
    print(f"VIDEO_FILTER={VIDEO_FILTER!r}: {pending['video_name'].nunique()} video(s) selected.")
else:
    pending = comparison_set.copy()

pending_judged = pending[pending["status"] == "judged"].copy()
pending_auto = pending[pending["status"] != "judged"].copy()

print("Report table exists:      ", bool(report_exists))
print("Rows to be replaced:      ", existing_rows if WRITE_TO_DB else f"{existing_rows} (kept, WRITE_TO_DB is False)")
print("Pending evaluation:       ", len(pending))
print("  -> sent to the judge:   ", len(pending_judged))
print("  -> resolved without LLM:", len(pending_auto))


Report table exists:       True
Rows to be replaced:       605
Pending evaluation:        605
  -> sent to the judge:    605
  -> resolved without LLM: 0


## 5. Load the judge

Only the selected backend is initialised, so a macOS run never needs a Groq key and a Kaggle run never needs MLX.
The local model is downloaded from Hugging Face on first use and cached afterwards.

In [9]:
groq_client = None
mlx_model = None
mlx_tokenizer = None
mlx_sampler = None
MLX_MAX_TOKENS = 300

if JUDGE_BACKEND == "groq":
    from groq import Groq

    groq_client = Groq(api_key=get_secret("GROQ_API_KEY", kaggle_name="groq"))

    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": "Reply exactly: judge connection working"}],
        temperature=0,
    )
    print("Groq:", response.choices[0].message.content)

else:
    from mlx_lm import generate as mlx_generate
    from mlx_lm import load as mlx_load
    from mlx_lm.sample_utils import make_sampler

    print(f"Loading local judge: {JUDGE_MODEL}", flush=True)
    mlx_model, mlx_tokenizer = mlx_load(JUDGE_MODEL)
    mlx_sampler = make_sampler(temp=0.0)
    print("Local judge ready.")

/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local judge: mlx-community/Qwen3-8B-4bit


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2053.46it/s]


Local judge ready.


## 6. The judge

Both backends get the same rules; only the shape differs.

| Score | Band | Meaning |
|---|---|---|
| 1.0 | Same action | An annotation names this activity, or describes the same action in other words |
| 0.9 | Same action | A close variant that is still this action, same object and direction |
| 0.8 | Implied | Not stated, but its result or an unmistakable sequence, or only the actions list names it |
| 0.7 | Partial evidence | Shown happening or its direct result (moving then parked), with its object or place named, but one element missing (completion, object, direction) (lowest covered score) |
| 0.6 | Start, approach or inference | The activity begins, the actors approach the object or place, or it is only inferred from what happens around it |
| 0.5 | Related | The right actors in a linked setting, static or unrelated to the action |
| 0.2–0.4 | Actors only | The right kind of actors appear doing something else (0.4 same object or area, 0.2 presence only) |
| 0.0–0.1 | Unrelated | Only the scene (0.1), or nothing related at all (0.0) |

The scale runs from 0.0 to 1.0 in steps of 0.1, and Part B's agreement judge uses the same bands. An activity
counts as **covered** from `COVERAGE_THRESHOLD` upwards (section 2, default `0.7`), and the
judge is told to be strict there: it needs evidence that the action happened, so starting, approaching or
being near something is at most 0.6, a description that fits several activities equally well (a car moving, a
person walking) is evidence for none of them, the described
actor must be the one doing the action ("the car drives off" does not show a person getting in), vague verbs
("interacts with", "moves around") are at most 0.5, a before-and-after state change (moving then parked) is
a direct result at 0.7, a turn without a direction is at most 0.7 for a left or right turn, and when torn it picks the lower score.
The rules still leave room for paraphrase: a described
direction change counts as a turn without the word "turn", leaning into a trunk counts for `person_opens_trunk`,
and the vehicle type or head-count never has to match. What is still refused is coverage by name alone (the
activity name is a hint about what to look for, not evidence), generic movement standing in for a specific
manoeuvre (slowing is not stopping, following a curve is not turning), and proximity standing in for an interaction.

Annotations produced with `videometa>=0.0.17` carry an `actions` list, the checklist phrases the annotator named
(`opens a vehicle door`, `vehicle reverses`, ...). It is shown to the judge next to the description as part of
the annotation: a listed action scores 1.0 when the description also describes it and 0.8 when only the list names it.

Timing is explicitly declared irrelevant in the prompt — the activity may be described anywhere in the video.


In [10]:
JUDGE_RULES = """coverage_score scale: 0.0 to 1.0 in steps of 0.1. An activity is covered from 0.7 upwards.
1.0 - same action: an annotation names this activity, or describes the same action in other words, with a
      synonym or a paraphrase ("gets out of the car" for person_exits_vehicle, "pulls out of the parking spot"
      for vehicle_starts, "swings around and heads back" for vehicle_makes_u_turn)
0.9 - same action, less direct: a close variant that is still this action, with the same object and direction
      ("climbs into the passenger side" for person_enters_vehicle)
0.8 - implied: no annotation states the action, but the text describes its result or an unmistakable sequence
      in which the same actor does it ("walks up to the SUV and the door swings open" for
      person_opens_vehicle_door, "walks through the doorway and is no longer visible" for
      person_exits_scene_through_structure), or only the annotation's actions list names it
0.7 - partial evidence: the text shows the action happening or its direct result, with the object or place
      involved named, but one element is missing or vague (completion, object or direction), and no other
      common activity explains it equally well ("walks into the doorway" for
      person_exits_scene_through_structure, "turns at the intersection" for vehicle_turns_left, "the moving
      car is later parked at the curb" for vehicle_stops)
0.6 - start, approach or inference: the right actors begin the activity or approach the object or place
      involved, or the action can only be inferred from what happens around it, but the text does not show it
      happening ("reaches for the door of the parked car" or "walks up to a parked car" for
      person_opens_vehicle_door, "slows down by the curb" for vehicle_stops, "walks to the car, which then drives off" for person_enters_vehicle, "heads toward the
      building entrance" for person_exits_scene_through_structure)
0.5 - related: the right kind of actors in a linked setting, static or doing something else ("a person stands
      next to a parked car" for person_opens_vehicle_door)
0.4 - actors only, same object or area: the right actors around the object or place involved, doing something
      unrelated
0.3 - actors only: the right kind of actors in the scene doing something unrelated
0.2 - presence only: the right kind of actors are merely visible, appearing or detected
0.1 - scene only: the annotations are about the same scene but not the right kind of actors
0.0 - unrelated: no annotation relates to this activity

Rules:
- Timing does not matter. The annotations are unordered and may describe the activity at any point in the video.
- How many annotations there are does not matter. One annotation describing the activity is enough.
- Wording does not matter. A synonym, paraphrase or everyday phrasing of the action is the same action (1.0).
- Be strict at the pass mark. A covered score (0.7 or higher) needs evidence in the text that the action
  happened, not that it began or could have happened: approaching, starting or being near something is at most
  0.6. When torn between 0.6 and 0.7, choose 0.6; when torn between 0.7 and 0.8, choose 0.7.
- Specific evidence only. If a description would fit several different activities equally well (a car moving,
  a person walking), it is not evidence for any of them: score it at most 0.6.
- The described actor must be the one doing the action. Do not credit an action by guessing what an unseen or
  unmentioned actor did ("the car drives off" does not show a person entering it).
- Vague verbs ("interacts with", "moves around", "is near", "gestures", "is active") are at most 0.5.
- A before-and-after state change of the same actor or object (moving then parked, closed then open) is a
  direct result: 0.7.
- Direction matters: a turn with no direction given is at most 0.7 for vehicle_turns_left or
  vehicle_turns_right, and the wrong direction is 0.0.
- A description that summarises the whole scene ("people walk around and cars pass by") is at most 0.4 for any
  specific activity.
- Read the event name and the description together. If the name claims an action that the description does not
  back up, score what the description shows.
- Never credit an action that only the activity name suggests.
- Movement: a described change of direction (turning at the intersection, into a driveway, heading back the
  way it came), coming to rest, pulling out from a parked position or backing up counts as turning, stopping,
  starting, reversing or a U-turn without that exact word. Slowing down without stopping is 0.6, following a
  curve in the road is not a turn, and moving straight through the frame is not a manoeuvre (0.3 to 0.5).
- Interaction: reaching into, leaning into, loading, climbing in or out or handing over counts for the matching
  activity. Holding counts for carrying only when the object is named. A door or trunk that is open with no
  person described at it is 0.6. Walking up to something is 0.6; standing near it with no interaction is 0.5.
- The actors should match in kind: an activity about a vehicle needs a vehicle in the annotation, one about a
  person needs a person. The exact vehicle type, colour or number of people does not matter.
- Judge the annotation text, not the activity name: the name tells you what to look for, not that it happened.
- An annotation's actions list, when present, is part of the annotation text: a listed action is the same
  action (1.0) when the description also describes it, and 0.8 when only the list names it."""


def numbered_annotations(annotations):
    lines = []
    for position, annotation in enumerate(annotations, start=1):
        line = f'{position}. event_name: "{annotation["event_name"]}" | description: "{annotation["description"]}"'
        actions = annotation.get("actions") or []
        if actions:
            line += f' | actions: {"; ".join(str(action) for action in actions)}'
        lines.append(line)
    return "\n".join(lines)


def build_groq_prompt(video_rows):
    """One call per video: the annotations once, then every MEVA activity type of that video."""
    first = video_rows.iloc[0]
    activity_types = list(video_rows["activity_type"])

    return f"""You are a fair evaluator checking whether the events an automatic annotation pipeline detected
in one CCTV video COVER the activities that human MEVA annotators recorded for that same video. Give credit
whenever the annotation text genuinely describes or clearly implies the activity, whatever the wording.

{JUDGE_RULES}

covered is true only when coverage_score >= {COVERAGE_THRESHOLD}.
covering_event_id is the event_id of the single best annotation when covered is true, otherwise null.

GENERATED ANNOTATIONS FOR VIDEO {first["camera_id"]} ({first["capture_start_local"]}):
{numbered_annotations(first["annotations"])}

MEVA ACTIVITY TYPES RECORDED IN THIS VIDEO:
{json.dumps(activity_types, indent=2)}

Judge every activity type in that list, in order. Return ONLY a valid JSON array, one object per activity type:

[
  {{
    "activity_type": "<the activity type, copied exactly>",
    "coverage_score": <one of 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0>,
    "covered": <true or false>,
    "covering_event_id": <event_id or null>,
    "reason": "<one short sentence>"
  }}
]
"""


def build_mlx_prompt(row):
    """One call per activity type, numbered annotations, index-based answer."""
    return f"""You are a fair evaluator. You are given ONE activity type that human annotators recorded in a
CCTV video, and all the events an automatic pipeline detected in that same video.

Decide whether ANY of the numbered annotations describes or clearly implies that activity. Give credit
whenever the text genuinely does, whatever the wording.

{JUDGE_RULES}

Answer with one JSON object and nothing else:
  "coverage_score": one of 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0
  "covered": true only when coverage_score is {COVERAGE_THRESHOLD} or higher
  "covering_index": the number of the annotation that describes the activity, or 0 if none does
  "reason": one short sentence explaining your score

MEVA ACTIVITY: {row["activity_type"]}

GENERATED ANNOTATIONS:
{numbered_annotations(row["annotations"])}

JSON object:"""

### 6.1 Normalising a verdict

Whatever the backend returns is reduced to the same five fields. The score is snapped to the nearest allowed
value, `covered` is derived from the score rather than taken from the model, and a `covering_event_id` is kept
only when it really is one of that video's annotations — a model asked for an event id will otherwise answer
with the MEVA activity name. A verdict with no usable score is discarded, leaving that pair pending.

In [11]:
ALLOWED_SCORES = tuple(step / 10 for step in range(11))   # 0.0, 0.1, ..., 1.0


def score_summary(scores, threshold, all_scores=None):
    """Score distribution of a group of verdicts, highest first, its pass rate, and the running pass rate of
    every verdict so far (`all_scores`, this group included)."""
    scores = [score for score in scores if score is not None]
    if not scores:
        return "    no verdicts in this group"
    counts = pd.Series(scores).round(1).value_counts().sort_index(ascending=False)
    passed = sum(score >= threshold for score in scores)
    spread = "  ".join(f"{score:.1f}x{count}" for score, count in counts.items())
    line = f"    scores: {spread}  | pass {passed}/{len(scores)} ({passed / len(scores):.0%})"
    if all_scores is not None:
        all_scores = [score for score in all_scores if score is not None]
        total_passed = sum(score >= threshold for score in all_scores)
        line += f"  | cumulative {total_passed}/{len(all_scores)} ({total_passed / len(all_scores):.0%})"
    return line


def normalise_verdict(pair_id, score, reason, covering_event_id, annotations):
    """Return a clean verdict record, or None when the response is unusable."""
    try:
        score = min(ALLOWED_SCORES, key=lambda allowed: abs(allowed - float(score)))
    except (TypeError, ValueError):
        return None

    covered = score >= COVERAGE_THRESHOLD

    valid_event_ids = {annotation["event_id"] for annotation in annotations}
    if not covered or covering_event_id not in valid_event_ids:
        covering_event_id = None

    return {
        "pair_id": pair_id,
        "coverage_score": score,
        "covered": covered,
        "covering_event_id": covering_event_id,
        "reason": str(reason or "").strip(),
    }

## 7. Parse the judge output

Both backends are asked for JSON. The helpers strip Markdown fences and any `<think>` block (Qwen3 emits
reasoning tags even with thinking disabled) before extracting the array or object from the response.

In [12]:
def strip_wrappers(response_text):
    """Remove reasoning tags and Markdown fences."""
    cleaned = re.sub(r"<think>.*?</think>", "", response_text, flags=re.DOTALL)
    cleaned = cleaned.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def parse_json_array(response_text):
    cleaned = strip_wrappers(response_text)

    if not cleaned.startswith("["):
        match = re.search(r"\[.*\]", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON array in judge response.")
        cleaned = match.group(0)

    parsed = json.loads(cleaned)

    if not isinstance(parsed, list):
        raise ValueError("Judge response is not a JSON array.")

    return parsed


def parse_json_object(response_text):
    match = re.search(r"\{.*?\}", strip_wrappers(response_text), re.DOTALL)

    if not match:
        raise ValueError("No JSON object in judge response.")

    return json.loads(match.group(0))

## 8. Judge the pending pairs

`judge_pending` dispatches to the selected backend: Groq one video per call, MLX one activity per call. Either
way each call is retried once when the response is not parsable, and anything still unusable is left pending
instead of being scored. Videos with no annotations never reach the judge.

In [13]:
def judge_groq_video(video_rows):
    """Judge every MEVA activity type of one video in a single hosted call."""
    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": build_groq_prompt(video_rows)}],
        temperature=0,
    )
    verdicts = parse_json_array(response.choices[0].message.content)

    # Map the judge's activity_type back to our pair, and ignore anything we did not ask about
    # (the output template carries a placeholder the model sometimes echoes).
    by_activity = {
        str(row["activity_type"]).strip().lower(): row
        for _, row in video_rows.iterrows()
    }
    annotations = video_rows.iloc[0]["annotations"]

    normalised = []
    for verdict in verdicts:
        row = by_activity.get(str(verdict.get("activity_type", "")).strip().lower())
        if row is None:
            continue

        clean = normalise_verdict(
            row["pair_id"],
            verdict.get("coverage_score"),
            verdict.get("reason"),
            verdict.get("covering_event_id"),
            annotations,
        )
        if clean:
            normalised.append(clean)

    return normalised


def judge_mlx_pair(row):
    """Judge one activity type with the local model."""
    output = mlx_generate(
        mlx_model,
        mlx_tokenizer,
        prompt=mlx_tokenizer.apply_chat_template(
            [{"role": "user", "content": build_mlx_prompt(row)}],
            add_generation_prompt=True,
            enable_thinking=False,
        ),
        max_tokens=MLX_MAX_TOKENS,
        sampler=mlx_sampler,
        verbose=False,
    )
    verdict = parse_json_object(output)

    # The local model answers with a position in the numbered list, not an event id.
    annotations = row["annotations"]
    index = verdict.get("covering_index")
    covering_event_id = None
    if isinstance(index, int) and 1 <= index <= len(annotations):
        covering_event_id = annotations[index - 1]["event_id"]

    clean = normalise_verdict(
        row["pair_id"],
        verdict.get("coverage_score"),
        verdict.get("reason"),
        covering_event_id,
        annotations,
    )
    if clean is None:
        raise ValueError(f"Unusable verdict: {verdict}")

    return [clean]


def judge_pending(pending_df):
    """Judge every pending pair with the selected backend, retrying each call once."""
    verdicts = []
    failures = []

    if JUDGE_BACKEND == "mlx":
        units = [row for _, row in pending_df.iterrows()]
        run = judge_mlx_pair
        describe = lambda unit: f"{unit['camera_id']} / {unit['activity_type']}"
    else:
        units = [
            group for _, group in pending_df.groupby(["camera_id", "capture_start_local"], sort=True)
        ]
        run = judge_groq_video
        describe = lambda unit: (
            f"{unit.iloc[0]['camera_id']} {unit.iloc[0]['capture_start_local']}"
            f" ({len(unit)} activity types)"
        )

    reported = 0   # verdicts already summarised in a progress line
    for position, unit in enumerate(units, start=1):
        label = f"{position}/{len(units)} {describe(unit)}"

        for attempt in (1, 2):
            try:
                verdicts.extend(run(unit))
                break
            except Exception as error:
                print(f"  {label} attempt {attempt} failed: {error}")
                if attempt == 2:
                    failures.append(label)

        if JUDGE_BACKEND == "groq" or position % 20 == 0 or position == len(units):
            print(f"Judged {label}", flush=True)
            all_scores = [v["coverage_score"] for v in verdicts]
            print(score_summary(all_scores[reported:], COVERAGE_THRESHOLD, all_scores), flush=True)
            reported = len(verdicts)

    return verdicts, failures


judgements, failed_calls = judge_pending(pending_judged)

print(f"\nPairs sent to the judge:   {len(pending_judged)}")
print(f"Verdicts returned:         {len(judgements)}")
print(f"Auto-resolved without LLM: {len(pending_auto)}")
if failed_calls:
    print("Failed calls:", failed_calls)

Judged 20/605 G326 / person_enters_scene_through_structure
    scores: 0.7x14  0.6x2  0.0x4  | pass 14/20 (70%)  | cumulative 14/20 (70%)
Judged 40/605 G328 / vehicle_starts
    scores: 1.0x2  0.7x10  0.6x4  0.5x3  0.0x1  | pass 12/20 (60%)  | cumulative 26/40 (65%)
Judged 60/605 G328 / vehicle_turns_left
    scores: 0.7x12  0.6x3  0.0x5  | pass 12/20 (60%)  | cumulative 38/60 (63%)
Judged 80/605 G328 / person_exits_scene_through_structure
    scores: 0.7x13  0.6x5  0.5x1  0.0x1  | pass 13/20 (65%)  | cumulative 51/80 (64%)
Judged 100/605 G329 / person_exits_scene_through_structure
    scores: 0.7x18  0.0x2  | pass 18/20 (90%)  | cumulative 69/100 (69%)
Judged 120/605 G331 / person_reads_document
    scores: 1.0x2  0.7x18  | pass 20/20 (100%)  | cumulative 89/120 (74%)
Judged 140/605 G336 / person_opens_vehicle_door
    scores: 1.0x5  0.7x13  0.6x1  0.0x1  | pass 18/20 (90%)  | cumulative 107/140 (76%)
Judged 160/605 G336 / vehicle_starts
    scores: 1.0x1  0.7x15  0.0x4  | pass 16/20 

## 9. Build the coverage report

The verdicts are merged back onto the pending pairs by `pair_id`, and the pairs that never went to the judge are
filled in: `no_annotations` becomes score `0.0` / not covered. A judged pair that came back without a usable verdict becomes `judge_missing` and stays pending.

In [14]:
judgements_df = pd.DataFrame(
    judgements,
    columns=["pair_id", "coverage_score", "covered", "covering_event_id", "reason"],
).drop_duplicates(subset="pair_id", keep="last")

coverage_report = pending.merge(judgements_df, on="pair_id", how="left")

no_annotations = coverage_report["status"] == "no_annotations"
coverage_report.loc[no_annotations, "coverage_score"] = 0.0
coverage_report.loc[no_annotations, "covered"] = False
coverage_report.loc[no_annotations, "reason"] = (
    "The pipeline generated no annotations for this video."
)

missing_verdict = (
    (coverage_report["status"] == "judged") & coverage_report["coverage_score"].isna()
)
coverage_report.loc[missing_verdict, "status"] = "judge_missing"

coverage_report["covered"] = coverage_report["covered"].astype("object")
coverage_report["judge_backend"] = JUDGE_BACKEND
coverage_report["judge_model"] = JUDGE_MODEL
coverage_report["evaluated_at"] = datetime.now(timezone.utc)

annotation_names = {
    annotation["event_id"]: annotation["event_name"]
    for annotations in coverage_report["annotations"]
    for annotation in annotations
}
coverage_report["covering_event_name"] = coverage_report["covering_event_id"].map(annotation_names)

# Object check against the one annotation credited with covering the activity.
annotations_by_id = {
    annotation["event_id"]: annotation
    for annotations in coverage_report["annotations"]
    for annotation in annotations
}


def covering_objects_missing(row):
    annotation = annotations_by_id.get(row["covering_event_id"])
    if annotation is None:
        return None  # nothing was credited, so there is nothing to check
    return objects_missing_from_annotation(row["required_objects"], annotation)


coverage_report["covering_objects_missing"] = coverage_report.apply(covering_objects_missing, axis=1)

coverage_report["gt_ids_text"] = coverage_report["gt_ids"].apply(
    lambda ids: ",".join(str(gt_id) for gt_id in (ids or []))
)
coverage_report["required_objects_text"] = coverage_report["required_objects"].apply(",".join)
coverage_report["covering_objects_missing_text"] = coverage_report["covering_objects_missing"].apply(
    lambda objs: None if objs is None else ",".join(objs)
)

print("Report rows:", len(coverage_report))
print(coverage_report["status"].value_counts().to_string())
print("Pairs without a verdict:", int(missing_verdict.sum()))

Report rows: 605
status
judged    605
Pairs without a verdict: 0


In [15]:
coverage_report[
    [
        "video_name",
        "camera_id",
        "activity_type",
        "n_meva_instances",
        "n_annotations",
        "coverage_score",
        "covered",
        "covering_event_name",
        "covering_objects_missing_text",
        "status",
        "reason",
    ]
].head(30)

,video_name,camera_id,activity_type,n_meva_instances,n_annotations,coverage_score,covered,covering_event_name,covering_objects_missing_text,status,reason
0,2018-03-07.11-10-00.11-15-00.school.G299,G299,person_talks_on_phone,1,7,0.7,True,Person sits down and uses a phone,,judged,The description of annotation 1 states that a ...
1,2018-03-07.11-10-00.11-15-00.school.G299,G299,person_texts_on_phone,3,7,0.7,True,Person sits down and uses a phone,,judged,The description of annotation 1 states that a ...
2,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_carries_heavy_object,1,16,0.7,True,Two people interact near a trash can,,judged,The first annotation describes a woman carryin...
3,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_enters_scene_through_structure,3,16,0.7,True,Two people interact near a trash can,,judged,The first annotation describes a woman enterin...
4,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_exits_scene_through_structure,4,16,0.7,True,Two people interact near a trash can,,judged,The description mentions 'both people standing...
5,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_talks_on_phone,1,16,0.7,True,Two women walk down the stairs together,,judged,The description of annotation 7 explicitly sta...
6,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_talks_to_person,1,16,0.7,True,Two people interact near a trash can,,judged,The first annotation describes two people inte...
7,2018-03-07.11-00-06.11-05-06.school.G300,G300,person_texts_on_phone,1,16,0.7,True,Two women walk down the stairs together,,judged,The description of annotation 7 explicitly sta...
8,2018-03-07.11-00-06.11-05-06.school.G300,G300,vehicle_reverses,1,16,0.0,False,NaN,NaN,judged,None of the annotations describe or imply that...
9,2018-03-07.11-00-06.11-05-06.school.G300,G300,vehicle_starts,1,16,0.0,False,NaN,NaN,judged,None of the annotations describe or imply that...


## 10. Coverage metrics

Two headline numbers:

* **Activity coverage** — of the (video, activity type) pairs MEVA recorded, how many the pipeline described.
  This is the event comparison itself: each kind of activity in each video counts once.
* **Instance-weighted coverage** — the same verdicts weighted by how many times MEVA annotated each activity, so
  a missed activity that happened nine times counts nine times. Useful as a second view; it is not a count match.

Numbers from the local MLX judge and from Groq are not interchangeable,
so every stored row carries its `judge_backend` and `judge_model` and the final summary groups by them.

In [16]:
evaluable = coverage_report[coverage_report["status"].isin(["judged", "no_annotations"])].copy()
evaluable["covered"] = evaluable["covered"].fillna(False).astype(bool)

pairs = len(evaluable)
covered_pairs = int(evaluable["covered"].sum())
instances = int(evaluable["n_meva_instances"].sum())
covered_instances = int(evaluable.loc[evaluable["covered"], "n_meva_instances"].sum())

print(f"MEVA activity types compared: {pairs}")
print(f"Covered by an annotation:     {covered_pairs}")
print(f"Activity coverage:            {covered_pairs / pairs:.1%}" if pairs else "n/a")
print()
print(f"MEVA instances behind them:   {instances}")
print(f"Instance-weighted coverage:   {covered_instances / instances:.1%}" if instances else "n/a")
print()
print("Misses by kind:")
print(f"  video had no annotations at all:     {int((evaluable['status'] == 'no_annotations').sum())}")
print(f"  annotations describe something else: "
      f"{int(((evaluable['status'] == 'judged') & ~evaluable['covered']).sum())}")


MEVA activity types compared: 605
Covered by an annotation:     487
Activity coverage:            80.5%

MEVA instances behind them:   2720
Instance-weighted coverage:   85.9%

Misses by kind:
  video had no annotations at all:     0
  annotations describe something else: 118


In [17]:
by_activity = (
    evaluable.groupby("activity_type")
    .agg(videos=("pair_id", "count"), covered=("covered", "sum"),
         meva_instances=("n_meva_instances", "sum"))
    .assign(coverage=lambda df: (df["covered"] / df["videos"]).round(3))
    .sort_values(["coverage", "videos"], ascending=[True, False])
)

print("Coverage by MEVA activity type (worst first):")
by_activity

Coverage by MEVA activity type (worst first):


,videos,covered,meva_instances,coverage
activity_type,,,,
person_embraces_person,7,1,18,0.143
person_talks_on_phone,20,7,35,0.350
vehicle_turns_left,25,12,99,0.480
person_talks_to_person,42,22,254,0.524
person_texts_on_phone,29,19,240,0.655
vehicle_reverses,18,12,32,0.667
person_closes_facility_door,6,4,8,0.667
vehicle_drops_off_person,3,2,3,0.667
vehicle_turns_right,29,20,118,0.690


In [18]:
by_video = (
    evaluable.groupby(["camera_id", "capture_start_local"])
    .agg(activity_types=("pair_id", "count"), covered=("covered", "sum"),
         annotations=("n_annotations", "max"))
    .assign(coverage=lambda df: (df["covered"] / df["activity_types"]).round(3))
    .sort_values("coverage")
)

print("Coverage by video (worst first):")
by_video

Coverage by video (worst first):


activity_types  covered  annotations  coverage
camera_id capture_start_local                                                
G301      2018-03-07 16:50:07               2        0            4     0.000
G506      2018-03-05 14:05:00               2        0            1     0.000
G505      2018-03-07 11:10:01               3        0           30     0.000
          2018-03-05 13:15:00               1        0           12     0.000
G300      2018-03-07 16:50:06               3        1           15     0.333
...                                       ...      ...          ...       ...
G421      2018-03-07 11:00:01              14       14          105     1.000
          2018-03-07 11:10:01              15       15          108     1.000
G423      2018-03-05 13:15:00               2        2            1     1.000
G420      2018-03-05 13:20:01               3        3            4     1.000
G299      2018-03-07 11:10:00               2        2            7     1.000

[70 rows x 4 columns]

In [19]:
missed = evaluable[~evaluable["covered"]][
    ["camera_id", "capture_start_local", "activity_type", "n_meva_instances",
     "n_annotations", "has_annotation_with_all_objects", "coverage_score", "reason"]
].sort_values(["camera_id", "activity_type"])

print(f"MEVA activity types the pipeline did not cover: {len(missed)}")
missed.head(40)

MEVA activity types the pipeline did not cover: 118


,camera_id,capture_start_local,activity_type,n_meva_instances,n_annotations,has_annotation_with_all_objects,coverage_score,reason
14,G300,2018-03-07 16:50:06,person_talks_to_person,2,15,True,0.0,None of the annotations describe or imply that...
15,G300,2018-03-07 16:50:06,person_texts_on_phone,1,15,True,0.0,None of the annotations describe or imply that...
8,G300,2018-03-07 11:00:06,vehicle_reverses,1,16,True,0.0,None of the annotations describe or imply that...
9,G300,2018-03-07 11:00:06,vehicle_starts,1,16,True,0.0,None of the annotations describe or imply that...
16,G301,2018-03-07 16:50:07,person_exits_scene_through_structure,1,4,True,0.6,The annotations describe a person standing nea...
17,G301,2018-03-07 16:50:07,person_opens_facility_door,1,4,True,0.6,None of the annotations describe the person op...
26,G326,2018-03-07 11:10:00,person_talks_on_phone,1,8,True,0.0,None of the annotations describe or imply that...
27,G326,2018-03-07 11:10:00,person_talks_to_person,1,8,True,0.6,None of the annotations describe or clearly im...
43,G328,2018-03-05 13:20:01,hand_interacts_with_person,2,44,True,0.0,None of the annotations describe or imply a ha...
61,G328,2018-03-07 11:00:01,hand_interacts_with_person,1,37,True,0.6,No annotation directly describes or implies a ...


### 10.1 Object check

For each MEVA event, did the generated annotations name every object the activity implies
(`person_opens_trunk` implies a person and a vehicle)? Two totals: events where the annotations
found all the implied objects, and events where they missed at least one.

In [20]:
found_all_objects = coverage_report["has_annotation_with_all_objects"]

print("Events where the annotations found all the objects:", int(found_all_objects.sum()))
print("Events where the annotations missed at least one:  ", int((~found_all_objects).sum()))

Events where the annotations found all the objects: 604
Events where the annotations missed at least one:   1


Same two totals restricted to the events the judge marked as covered, checking only the single annotation it
credited. This is the stricter view: it asks whether the annotation that supposedly describes the activity names
the right objects.

In [21]:
credited = coverage_report[coverage_report["covering_objects_missing"].notna()]
credited_missing = credited["covering_objects_missing"].apply(len) > 0

print("Credited annotations checked:              ", len(credited))
print("  found all the objects:                  ", int((~credited_missing).sum()))
print("  missed at least one:                    ", int(credited_missing.sum()))

Credited annotations checked:               453
  found all the objects:                   453
  missed at least one:                     0


In [22]:
if credited_missing.any():
    print("Events whose credited annotation missed an implied object:")
    display(
        credited[credited_missing][
            [
                "camera_id",
                "activity_type",
                "required_objects_text",
                "covering_objects_missing_text",
                "covering_event_name",
                "judge_reason" if "judge_reason" in credited.columns else "reason",
            ]
        ].sort_values(["camera_id", "activity_type"])
    )
else:
    print("No credited annotation missed an implied object.")

No credited annotation missed an implied object.


### 10.2 Judge sanity check

Two signatures of a judge that is rubber-stamping instead of reading, both seen for real with the local 0.6B
model:

* **One annotation credited with many different activities in the same video.** "A car appears in the
  middle-right" cannot be what describes `person_enters_vehicle`, `vehicle_reverses` *and*
  `person_enters_scene_through_structure`.
* **A flat score distribution**, especially almost everything at one value such as 0.8, which is the model repeating the
  middle of the scale rather than judging.

These checks do not change any verdict; they tell you whether the coverage number above is worth reporting.

In [23]:
judged_only = evaluable[evaluable["status"] == "judged"]

print("Score distribution over judged pairs:")
print(judged_only["coverage_score"].value_counts().sort_index().to_string())

if len(judged_only):
    counts = judged_only["coverage_score"].value_counts()
    levels_used = len(counts)

    print(f"\nScore levels used: {levels_used} of {len(ALLOWED_SCORES)}")

    if levels_used <= 2:
        print(
            f"WARNING: the judge only ever answered {sorted(counts.index)}. Two levels can mean the"
            " pipeline really is uniformly good or bad, but with a small model it usually means the"
            " judge is picking a fixed answer rather than grading. Compare against another judge"
            " before reporting."
        )

reused = (
    evaluable[evaluable["covered"] & evaluable["covering_event_id"].notna()]
    .groupby(["camera_id", "capture_start_local", "covering_event_name"])
    .agg(activity_types=("activity_type", "nunique"),
         activities=("activity_type", lambda values: ", ".join(sorted(values))))
    .reset_index()
    .sort_values("activity_types", ascending=False)
)
reused = reused[reused["activity_types"] >= 3]

if len(reused):
    print(
        f"\nWARNING: {len(reused)} annotation(s) are credited with covering 3 or more different MEVA"
        " activities in the same video. Check these before reporting the coverage number:"
    )
    display(reused.head(10))
else:
    print("\nNo annotation is credited with covering 3 or more different activities in one video.")

Score distribution over judged pairs:
coverage_score
0.0     72
0.5      8
0.6     38
0.7    440
0.8      1
0.9      4
1.0     42

Score levels used: 7 of 11



,camera_id,capture_start_local,covering_event_name,activity_types,activities
179,G424,2018-03-05 13:20:00,A person exits a white SUV and walks away,15,"hand_interacts_with_person, person_carries_hea..."
159,G421,2018-03-07 11:10:01,Man walks toward a group near the doors,9,"hand_interacts_with_person, person_embraces_pe..."
77,G340,2018-03-05 13:15:00,Man opens the trunk of a white SUV and retriev...,9,"person_carries_heavy_object, person_closes_tru..."
106,G341,2018-03-07 11:00:04,Person walks toward a parked car,8,"person_carries_heavy_object, person_closes_tru..."
187,G436,2018-03-07 11:10:00,Person walks toward a parked white SUV,8,"person_carries_heavy_object, person_enters_sce..."
112,G341,2018-03-07 11:05:04,Person walks toward the building entrance,7,"person_carries_heavy_object, person_exits_scen..."
176,G424,2018-03-05 13:15:00,Person walks toward a parked white SUV,7,"person_carries_heavy_object, person_closes_veh..."
196,G506,2018-03-05 13:15:00,Man opens the trunk of a white SUV,6,"person_closes_trunk, person_opens_trunk, perso..."
76,G339,2018-03-07 11:10:07,Person walks toward a parked car,6,"hand_interacts_with_person, person_enters_scen..."
24,G328,2018-03-07 11:05:01,A person walks toward a parked van,6,"person_enters_scene_through_structure, person_..."


## 11. Write to `bronze.meva_event_coverage`

Without `VIDEO_FILTER` the table is dropped and rebuilt from this run's report, so it only ever holds the newest
verdicts — old rows from earlier runs or other judge models are removed. With `VIDEO_FILTER` set, only the
matching videos' rows are deleted and rewritten, in one transaction, and every other row stays. Only pairs
carrying a verdict are written; `judge_missing` rows are left out and a re-run retries them. Set
`WRITE_TO_DB = False` in section 2 to keep the report local and leave the stored table untouched.


In [24]:
import csv
import io

from sqlalchemy import BigInteger, Boolean, DateTime, Float, Integer, Text


def psql_insert_copy(table, conn, keys, data_iter):
    """to_sql insert method that streams rows with COPY instead of a multi-row INSERT.

    One INSERT carrying every row (~5,600 rows, 120k+ bind parameters) hits Supabase's 2-minute
    statement timeout, and chunked INSERTs take ~18 s per 500 rows over the pooler. COPY writes the
    same rows in well under a minute. Empty CSV fields are read back as NULL.
    """
    buffer = io.StringIO()
    csv.writer(buffer).writerows(data_iter)
    buffer.seek(0)

    columns = ", ".join(f'"{key}"' for key in keys)
    name = f"{table.schema}.{table.name}" if table.schema else table.name
    with conn.connection.cursor() as cursor:
        cursor.copy_expert(f"COPY {name} ({columns}) FROM STDIN WITH CSV", buffer)


REPORT_COLUMNS = [
    "pair_id",
    "camera_id",
    "capture_start_local",
    "video_id",
    "video_name",
    "activity_type",
    "n_meva_instances",
    "gt_ids_text",
    "required_objects_text",
    "covering_objects_missing_text",
    "n_annotations",
    "has_annotation_with_all_objects",
    "n_annotations_with_all_objects",
    "coverage_score",
    "covered",
    "covering_event_id",
    "covering_event_name",
    "reason",
    "status",
    "judge_backend",
    "judge_model",
    "evaluated_at",
]

REPORT_RENAMES = {
    "gt_ids_text": "gt_ids",
    "required_objects_text": "required_objects",
    "covering_objects_missing_text": "covering_objects_missing",
    "reason": "judge_reason",
}

# Explicit types: `covered` carries NULLs for unevaluable pairs, and without this
# to_sql would create it as TEXT the first time the table is built.
REPORT_DTYPES = {
    "pair_id": Text(),
    "camera_id": Text(),
    "capture_start_local": DateTime(),
    "video_id": Text(),
    "video_name": Text(),
    "activity_type": Text(),
    "n_meva_instances": Integer(),
    "gt_ids": Text(),
    "required_objects": Text(),
    "covering_objects_missing": Text(),
    "n_annotations": Integer(),
    "has_annotation_with_all_objects": Boolean(),
    "n_annotations_with_all_objects": Integer(),
    "coverage_score": Float(),
    "covered": Boolean(),
    "covering_event_id": Text(),
    "covering_event_name": Text(),
    "judge_reason": Text(),
    "status": Text(),
    "judge_backend": Text(),
    "judge_model": Text(),
    "evaluated_at": DateTime(timezone=True),
}

writable = (
    coverage_report[coverage_report["status"] != "judge_missing"][REPORT_COLUMNS]
    .rename(columns=REPORT_RENAMES)
    .copy()
)

# NaN is not a boolean; send a real NULL for pairs that were never scored.
writable["covered"] = writable["covered"].where(writable["covered"].notna(), None)
writable["coverage_score"] = writable["coverage_score"].astype("float64")

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(writable)} rows kept local, nothing written.")
elif writable.empty:
    print("No coverage rows with a verdict - the stored table is left untouched.")
elif VIDEO_FILTER:
    # Replace only the filtered videos' rows: delete and append inside one transaction, so a failed
    # insert rolls the delete back too.
    with engine.begin() as conn:
        deleted = 0
        if report_exists:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{COVERAGE_TABLE} WHERE video_name LIKE :pattern;"),
                {"pattern": f"%{VIDEO_FILTER}%"},
            ).rowcount
        writable.to_sql(
            COVERAGE_TABLE,
            conn,
            schema="bronze",
            if_exists="append",
            index=False,
            method=psql_insert_copy,
            dtype=REPORT_DTYPES,
        )
    print(f"Replaced rows for {VIDEO_FILTER!r} in bronze.{COVERAGE_TABLE}: "
          f"{deleted} old rows removed, {len(writable)} rows written.")
else:
    # Drop and recreate the table with this run's rows only. pandas runs this in one transaction,
    # so a failed insert leaves the previous table in place.
    writable.to_sql(
        COVERAGE_TABLE,
        engine,
        schema="bronze",
        if_exists="replace",
        index=False,
        method=psql_insert_copy,
        dtype=REPORT_DTYPES,
    )
    print(f"Replaced bronze.{COVERAGE_TABLE}: {existing_rows} old rows removed, {len(writable)} rows written.")

held_back = int((coverage_report["status"] == "judge_missing").sum())
if held_back:
    print(f"Held back {held_back} pairs without a verdict; re-run the notebook to retry them.")

Replaced bronze.meva_event_coverage: 605 old rows removed, 605 rows written.


In [25]:
if report_exists or (WRITE_TO_DB and not writable.empty):
    summary = pd.read_sql(
        text(f"""
        WITH latest AS (
            -- one row per pair and judge model: the newest verdict wins
            SELECT DISTINCT ON (pair_id, judge_model) *
            FROM bronze.{COVERAGE_TABLE}
            WHERE status IN ('judged', 'no_annotations')
            ORDER BY pair_id, judge_model, evaluated_at DESC
        )
        SELECT judge_backend,
               judge_model,
               COUNT(*) AS activity_types,
               SUM(CASE WHEN covered THEN 1 ELSE 0 END) AS covered,
               ROUND(AVG(CASE WHEN covered THEN 1.0 ELSE 0.0 END), 3) AS coverage
        FROM latest
        GROUP BY judge_backend, judge_model
        ORDER BY judge_backend;
        """),
        engine,
    )
    print(f"bronze.{COVERAGE_TABLE} contents by judge:")
    display(summary)
else:
    print(f"bronze.{COVERAGE_TABLE} does not exist yet.")

bronze.meva_event_coverage contents by judge:


,judge_backend,judge_model,activity_types,covered,coverage
0,mlx,mlx-community/Qwen3-8B-4bit,605,487,0.805


# Part B — Agreement: do the generated events describe what MEVA recorded at that time?

The opposite direction. Every generated event is matched to the MEVA activities whose frame ranges overlap its
window in the same video, exactly as the earlier matching stage did (`camera_id` + capture time, frame overlap,
activities in chronological order, each turned into a fixed sentence). The judge then compares the annotation
text with those reference sentences on the same 0.0-1.0 scale as Part A, in steps of 0.1: same action (0.9-1.0), implied (0.8),
plausible (0.7), related (0.5-0.6), actor only (0.2-0.4), unrelated or contradicted (0.0-0.1).

An event with no overlapping MEVA activity is recorded as `no_reference` and never sent to the judge; a video
with no ground truth at all gives `has_ground_truth = false`.

## 12. Match generated events to MEVA activities

`VIDEO_FILTER` applies here too: with it set only the matching videos' events are judged and only their rows
for this prompt version are replaced.

The pairs are also written to `bronze.matched_pairs`, the table the QA notebook reads, so it keeps working on
fresh data: all rows are replaced on a full run, only the filtered videos' rows with `VIDEO_FILTER`.


In [26]:
# One fixed sentence per MEVA activity type, the wording the earlier matching stage used.
MEVA_SENTENCES = {
    "hand_interacts_with_person": "a person's hand interacts with another person",
    "person_carries_heavy_object": "a person carries a heavy object",
    "person_closes_facility_door": "a person closes a facility door",
    "person_closes_trunk": "a person closes a trunk",
    "person_closes_vehicle_door": "a person closes a vehicle door",
    "person_embraces_person": "a person embraces another person",
    "person_enters_scene_through_structure": "a person enters the scene through a structure",
    "person_enters_vehicle": "a person enters a vehicle",
    "person_exits_scene_through_structure": "a person exits the scene through a structure",
    "person_exits_vehicle": "a person exits a vehicle",
    "person_interacts_with_laptop": "a person interacts with a laptop",
    "person_loads_vehicle": "a person loads a vehicle",
    "person_opens_facility_door": "a person opens a facility door",
    "person_opens_trunk": "a person opens a trunk",
    "person_opens_vehicle_door": "a person opens a vehicle door",
    "person_picks_up_object": "a person picks up an object",
    "person_purchases": "a person purchases something",
    "person_puts_down_object": "a person puts down an object",
    "person_reads_document": "a person reads a document",
    "person_rides_bicycle": "a person rides a bicycle",
    "person_sits_down": "a person sits down",
    "person_stands_up": "a person stands up",
    "person_steals_object": "a person steals an object",
    "person_talks_on_phone": "a person talks on a phone",
    "person_talks_to_person": "a person talks to another person",
    "person_texts_on_phone": "a person texts on a phone",
    "person_transfers_object": "a person transfers an object to another person",
    "person_unloads_vehicle": "a person unloads a vehicle",
    "vehicle_drops_off_person": "a vehicle drops off a person",
    "vehicle_makes_u_turn": "a vehicle makes a U-turn",
    "vehicle_picks_up_person": "a vehicle picks up a person",
    "vehicle_reverses": "a vehicle reverses",
    "vehicle_starts": "a vehicle starts moving",
    "vehicle_stops": "a vehicle stops",
    "vehicle_turns_left": "a vehicle turns left",
    "vehicle_turns_right": "a vehicle turns right",
}
IGNORED_MEVA_TYPES = ("empty_37",)   # MEVA placeholder rows, not activities


def meva_sentence(activity_type):
    return MEVA_SENTENCES.get(activity_type, "a " + activity_type.replace("_", " "))


matching_sql = text(f"""
SELECT
    e.event_id,
    e.video_id,
    v.video_name,
    v.camera_id,
    e.event_name    AS model_event_name,
    e.description   AS model_description,
    e.start_frame   AS event_start_frame,
    e.end_frame     AS event_end_frame,
    array_agg(g.activity_type ORDER BY g.start_frame, g.activity_type) FILTER (WHERE g.id IS NOT NULL) AS matched_types,
    EXISTS (SELECT 1 FROM bronze.ground_truth gt WHERE gt.video_id = e.video_id) AS has_ground_truth
FROM bronze.events e
JOIN bronze.videos v ON v.video_id = e.video_id
LEFT JOIN bronze.ground_truth g
    ON g.video_id = e.video_id
   AND NOT (g.activity_type = ANY(:ignored))
   AND g.start_frame <= e.end_frame + :tolerance
   AND g.end_frame   >= e.start_frame - :tolerance
WHERE e.start_frame IS NOT NULL AND e.end_frame IS NOT NULL
  AND (CAST(:pattern AS text) IS NULL OR v.video_name LIKE :pattern)
GROUP BY e.event_id, e.video_id, v.video_name, v.camera_id, e.event_name, e.description, e.start_frame, e.end_frame
ORDER BY v.video_name, e.start_frame, e.event_id;
""")

matched_events = pd.read_sql(
    matching_sql,
    engine,
    params={
        "ignored": list(IGNORED_MEVA_TYPES),
        "tolerance": FRAME_TOLERANCE,
        "pattern": f"%{VIDEO_FILTER}%" if VIDEO_FILTER else None,
    },
)


def unique_in_order(values):
    seen, ordered = set(), []
    for value in values or []:
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered


matched_events["matched_types"] = matched_events["matched_types"].apply(unique_in_order)
matched_events["reference_list"] = matched_events["matched_types"].apply(lambda types: [meva_sentence(t) for t in types])
matched_events["meva_reference"] = matched_events["reference_list"].apply(
    lambda refs: "; ".join(refs) if refs else "(no matching MEVA ground truth found)"
)
matched_events["n_meva_activities_matched"] = matched_events["matched_types"].apply(len)
# A stable integer id per event, so (id, prompt_version) stays unique the way the QA notebook's table expects.
matched_events["id"] = matched_events["event_id"].apply(lambda event_id: int(str(event_id)[:15], 16))

matched_events["agreement_status"] = "judged"
matched_events.loc[matched_events["n_meva_activities_matched"] == 0, "agreement_status"] = "no_reference"

print("Generated events selected:", len(matched_events), "in", matched_events["video_id"].nunique(), "video(s)")
print(matched_events["agreement_status"].value_counts().to_string())
print("Events in videos with no ground truth at all:", int((~matched_events["has_ground_truth"]).sum()))
matched_events[["video_name", "model_event_name", "event_start_frame", "event_end_frame",
                "n_meva_activities_matched", "meva_reference", "agreement_status"]].head(12)


Generated events selected: 1955 in 71 video(s)
agreement_status
judged          1788
no_reference     167
Events in videos with no ground truth at all: 1


,video_name,model_event_name,event_start_frame,event_end_frame,n_meva_activities_matched,meva_reference,agreement_status
0,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,939,1020,1,a vehicle reverses,judged
1,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,960,1041,1,a vehicle reverses,judged
2,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,1922,1980,0,(no matching MEVA ground truth found),no_reference
3,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,1922,2063,0,(no matching MEVA ground truth found),no_reference
4,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV enters the frame and parks,2384,2460,1,a vehicle stops,judged
5,2018-03-05.13-15-00.13-20-00.bus.G340,Man walks toward a parked white SUV,2400,2700,1,a vehicle stops,judged
6,2018-03-05.13-15-00.13-20-00.bus.G340,Man opens the trunk of a white SUV and retriev...,2640,2940,4,a person opens a trunk; a person exits a vehic...,judged
7,2018-03-05.13-15-00.13-20-00.bus.G340,Man pulls a suitcase from a white SUV,2880,3180,9,a person exits a vehicle; a person unloads a v...,judged
8,2018-03-05.13-15-00.13-20-00.bus.G340,Two men interact near a white SUV,3120,3420,6,a person talks to another person; a person clo...,judged
9,2018-03-05.13-15-00.13-20-00.bus.G340,Two men interact near a white SUV while handli...,3360,3660,5,a person talks to another person; a person car...,judged


In [27]:
MATCHED_PAIRS_COLUMNS = [
    "id", "event_id", "video_id", "camera_id", "model_event_name", "model_description", "event_start_frame",
    "event_end_frame", "meva_reference", "n_meva_activities_matched", "created_at", "has_ground_truth",
]
MATCHED_PAIRS_DDL = f"""
CREATE TABLE IF NOT EXISTS bronze.{MATCHED_PAIRS_TABLE} (
    id BIGINT PRIMARY KEY,
    event_id TEXT,
    camera_id TEXT,
    model_event_name TEXT,
    model_description TEXT,
    event_start_frame INTEGER,
    event_end_frame INTEGER,
    meva_reference TEXT,
    n_meva_activities_matched INTEGER,
    created_at TIMESTAMPTZ DEFAULT now(),
    has_ground_truth BOOLEAN
);
ALTER TABLE bronze.{MATCHED_PAIRS_TABLE} ADD COLUMN IF NOT EXISTS video_id TEXT;
"""

matched_pairs_rows = matched_events[MATCHED_PAIRS_COLUMNS[:2] + MATCHED_PAIRS_COLUMNS[3:-2] + ["has_ground_truth"]].copy()
matched_pairs_rows["video_id"] = matched_events["video_id"]
matched_pairs_rows["created_at"] = datetime.now(timezone.utc)
matched_pairs_rows = matched_pairs_rows[MATCHED_PAIRS_COLUMNS]

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(matched_pairs_rows)} matched pairs kept local, nothing written.")
elif matched_pairs_rows.empty:
    print("No matched pairs to write.")
else:
    # Same rule as every other table: replace the filtered videos' rows, or everything, in one transaction.
    with engine.begin() as conn:
        for statement in MATCHED_PAIRS_DDL.strip().split(";"):
            if statement.strip():
                conn.execute(text(statement))
        if VIDEO_FILTER:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{MATCHED_PAIRS_TABLE} WHERE event_id = ANY(:ids) OR video_id = ANY(:videos);"),
                {"ids": list(matched_events["event_id"]), "videos": list(matched_events["video_id"].unique())},
            ).rowcount
        else:
            deleted = conn.execute(text(f"DELETE FROM bronze.{MATCHED_PAIRS_TABLE};")).rowcount
        matched_pairs_rows.to_sql(
            MATCHED_PAIRS_TABLE, conn, schema="bronze", if_exists="append", index=False, method=psql_insert_copy,
        )
    print(f"bronze.{MATCHED_PAIRS_TABLE}: {deleted} old rows removed, {len(matched_pairs_rows)} rows written.")


bronze.matched_pairs: 1955 old rows removed, 1955 rows written.


## 13. The agreement judge

The rubric (`v4_unified_scale_judge`) uses the same 0.0-1.0 scale as Part A, in steps of 0.1, so a score means the
same thing in both tables: 0.9-1.0 the same action in any wording, 0.8 implied, 0.7 partial evidence, all counting as a
match from `AGREEMENT_THRESHOLD = 0.7`. Below that, 0.5-0.6 is a related situation, 0.2-0.4 the right actor doing
something unrelated (movement or position only), 0.0-0.1 the wrong actor or a contradiction. A match needs evidence in
the text that the action happened: starting or approaching it is at most 0.6, a description that fits several
activities equally well counts for none of them, an action inferred for an unmentioned actor is not credited, and the judge picks the lower score when torn. Earlier versions (`v2.x`, `v3`) stay in the table for comparison. The
hosted judge takes `AGREEMENT_BATCH_SIZE` events per call with a strict JSON schema; the local judge takes one
event per call. `llm_match` is computed in Python as `score >= AGREEMENT_THRESHOLD`.


In [28]:
AGREEMENT_RULES = """You are evaluating generated CCTV event annotations against MEVA reference annotations.

The generated annotation (name + description) usually describes appearance, position and movement.
The MEVA reference lists one or more fixed activities joined by ";". Compare the annotation with EACH
reference activity and give the score of the BEST-matching one.

Scores run from 0.0 to 1.0 in steps of 0.1. A score of 0.7 or higher counts as a match.
Apply these steps in order and stop at the first that applies:

STEP 1 -> 0.0 to 0.1 (unrelated)
  0.0: the description explicitly states the OPPOSITE of the same kind of action in the reference:
  exits vs enters (a vehicle or the scene), opens vs closes, picks up vs puts down, loads vs unloads,
  turns left vs turns right.
  0.1: no reference activity has the same kind of actor as the description.
  Actor = the subject of the activity: "a vehicle ..." is a vehicle activity, "a person ..." is a person
  activity (including person-vehicle interactions like "a person opens a vehicle door").
  If the description covers both people and vehicles, both actor types count.
  NOT a contradiction: an actor being idle, standing, parked or simply present. Other people or vehicles
  in the scene may be doing the reference activity.

STEP 2 -> 0.9 to 1.0 (same action)
  1.0: the description states a reference action, in the same words or in other words (a synonym or a
  paraphrase), for example:
    "curves around and heads back the way it came" -> makes a U-turn
    "pulls into a space and comes to rest" -> stops
    "gets out of the car" -> exits a vehicle
  0.9: a close variant that is still the reference action, with the same object and direction ("climbs into
  the passenger side" -> enters a vehicle).

STEP 3 -> 0.8 (implied)
  The description does not state a reference action, but it describes its result or an unmistakable
  sequence in which the same actor does it, for example:
    "walks up to the SUV and the door swings open" -> opens a vehicle door
    "walks through the doorway and is no longer visible" -> exits the scene through a structure

STEP 4 -> 0.7 (partial evidence)
  Same actor type, and the description shows a reference action happening or its direct result, with the
  object or place involved named, but one element is missing or vague (completion, object or direction),
  and no other common activity explains it equally well, for example:
    "walks into the doorway" -> exits the scene through a structure
    "turns at the intersection" -> turns left
    "the moving car is later parked at the curb" -> stops

  Be strict at the pass mark. A match (0.7 or higher) needs evidence that the action happened, not that it
  began or could have happened: starting, approaching or heading toward the object or place involved is at
  most 0.6. A description that would fit several different activities equally well (a car moving, a person
  walking) is not evidence for any of them. The described actor must be the one doing the action: do not
  credit an action by guessing what an unseen or unmentioned actor did ("the car drives off" does not show
  a person entering it). Vague verbs ("interacts with", "moves around", "is near", "gestures") are at most
  0.5. A before-and-after state change of the same actor or object (moving then parked, closed then
  open) is a direct result, 0.7. A description that summarises the whole scene ("people walk around
  and cars pass by") is at most 0.4. If the generated name claims an action the description does not back up, score what the description
  shows. When torn between 0.6 and 0.7, choose 0.6; when torn between 0.7 and 0.8, choose 0.7.

STEP 5 -> 0.5 to 0.6 (related)
  Same actor type, in a situation linked to a reference action, but the text does not show the action
  happening. 0.6 when the actor starts it, approaches the object or place involved, or the action can only
  be inferred from what happens around it ("reaches for the door of the parked car" or "walks up to a parked
  car" for "opens a vehicle door", "slows down by the curb" for "stops", "walks to the car, which then drives
  off" for "enters a vehicle"); 0.5 when the actor is static or doing something else, for example a person standing beside a
  parked car.

STEP 6 -> 0.2 to 0.4 (actor only)
  Same actor type, but the description only gives appearance, position, presence or movement unrelated to
  any reference action. 0.4 when it is around the object or place involved, 0.3 for unrelated activity,
  0.2 when the actor is merely visible, appearing or detected.

Activity rules (what counts as describing or implying the change):
- "starts moving": a stationary vehicle begins to move; pulling out or driving off from a parked position
  count. Driving through the frame the whole time is not starting.
- "stops": a moving vehicle comes to rest; pulling in and parking count. Slowing down without stopping is
  0.6. Parked or stationary the whole time is not stopping.
- turns / U-turn: a described change of direction; a turn with no direction given is at most 0.7 for
  "turns left" or "turns right"; turning at an intersection, into a lane or driveway, or
  heading back the way it came count. Following a curve in the road or moving straight from one side of the
  frame to the other is not a turn.
- "reverses": the vehicle backs up or backs out.
- "enters the scene through a structure": a person comes out of a door/building into view.
- "exits the scene through a structure": a person goes into a door/building and leaves view; walking
  through the doorway and disappearing implies it (0.8); walking into the doorway is partial evidence
  (0.7); heading toward one is 0.6. Standing near an entrance is related (0.5).
- carries / picks up / puts down / loads / unloads: the object must be named ("holding a bag", "with a box").
- "appears", "is visible", "is detected", "enters the frame" on their own are not activities (0.2).

Evaluate each annotation independently. Return one result per id.
reason: at most 15 words; name the reference activity compared and the step applied,
e.g. "vehicle stops: same actor, moving only, no stop described (step 6)"."""

AGREEMENT_SCORE_LABELS = [f"{score:.1f}" for score in reversed(ALLOWED_SCORES)]   # "1.0", "0.9", ..., "0.0"
AGREEMENT_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "score": {"type": "string", "enum": AGREEMENT_SCORE_LABELS},
                    "reason": {"type": "string"},
                },
                "required": ["id", "score", "reason"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["results"],
    "additionalProperties": False,
}
AGREEMENT_MAX_RETRIES = 4
AGREEMENT_MAX_WAIT_SECONDS = 120
AGREEMENT_SLEEP_BETWEEN_CALLS = 2.0


def annotation_payload(row):
    return {
        "id": int(row["id"]),
        "generated_name": row["model_event_name"],
        "generated_description": row["model_description"],
        "meva_reference": "; ".join(row["reference_list"]),
    }


def snap_agreement_score(value):
    """Nearest allowed score, or None when the value is unusable."""
    try:
        return min(ALLOWED_SCORES, key=lambda allowed: abs(allowed - float(value)))
    except (TypeError, ValueError):
        return None


def agreement_groq_batch(batch_df):
    """One hosted call for a batch of events, strict JSON output; retried on rate limits and timeouts."""
    import time

    import groq

    retryable = tuple(
        error for error in (
            getattr(groq, "RateLimitError", None), getattr(groq, "APITimeoutError", None),
            getattr(groq, "APIConnectionError", None), getattr(groq, "InternalServerError", None),
        ) if error is not None
    )
    payload = {"annotations": [annotation_payload(row) for _, row in batch_df.iterrows()]}
    for attempt in range(1, AGREEMENT_MAX_RETRIES + 1):
        try:
            response = groq_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": AGREEMENT_RULES},
                    {"role": "user", "content": json.dumps(payload, separators=(",", ":"))},
                ],
                temperature=0,
                reasoning_effort="low",
                max_completion_tokens=4096,
                response_format={
                    "type": "json_schema",
                    "json_schema": {"name": "annotation_judgements", "strict": True, "schema": AGREEMENT_RESPONSE_SCHEMA},
                },
            )
            break
        except retryable as error:
            headers = getattr(getattr(error, "response", None), "headers", None) or {}
            try:
                wait = float(headers.get("retry-after")) if headers.get("retry-after") else min(60, 2 ** attempt)
            except ValueError:
                wait = min(60, 2 ** attempt)
            if wait > AGREEMENT_MAX_WAIT_SECONDS:
                raise RuntimeError(f"Groq asked to wait {wait:.0f}s (likely a daily limit); rerun later.") from error
            if attempt == AGREEMENT_MAX_RETRIES:
                raise
            print(f"    {type(error).__name__}; retry {attempt}/{AGREEMENT_MAX_RETRIES - 1} in {wait:.0f}s", flush=True)
            time.sleep(wait)
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise ValueError("Response truncated; lower AGREEMENT_BATCH_SIZE.")
    results = json.loads(choice.message.content).get("results", [])
    expected = {int(row["id"]) for _, row in batch_df.iterrows()}
    verdicts = {}
    for item in results:
        try:
            item_id = int(item.get("id"))
        except (TypeError, ValueError):
            continue
        score = snap_agreement_score(item.get("score"))
        if item_id in expected and item_id not in verdicts and score is not None:
            verdicts[item_id] = {"score": score, "reason": str(item.get("reason") or "").strip()}
    return verdicts


def agreement_mlx_event(row):
    """One local call per event: the same rubric, one annotation, a JSON object back."""
    prompt = (
        AGREEMENT_RULES
        + "\n\nANNOTATION TO EVALUATE:\n"
        + json.dumps(annotation_payload(row), indent=2)
        + "\n\nAnswer with one JSON object and nothing else:\n"
        + '  "score": one of 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0\n'
        + '  "reason": one short sentence naming the reference activity compared and the step applied\n\n'
        + "JSON object:"
    )
    output = mlx_generate(
        mlx_model, mlx_tokenizer,
        prompt=mlx_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], add_generation_prompt=True, enable_thinking=False,
        ),
        max_tokens=MLX_MAX_TOKENS, sampler=mlx_sampler, verbose=False,
    )
    verdict = parse_json_object(output)
    score = snap_agreement_score(verdict.get("score"))
    if score is None:
        raise ValueError(f"Unusable verdict: {verdict}")
    return {int(row["id"]): {"score": score, "reason": str(verdict.get("reason") or "").strip()}}


def judge_agreement(events_df):
    """Judge every event with the selected backend; a failed batch is split in halves down to single events."""
    import time

    verdicts, failures = {}, {}

    def run_batch(batch):
        try:
            if JUDGE_BACKEND == "groq":
                got = agreement_groq_batch(batch)
            else:
                got = {}
                for _, row in batch.iterrows():
                    got.update(agreement_mlx_event(row))
        except RuntimeError:
            raise
        except Exception as error:
            got, problem = {}, f"{type(error).__name__}: {error}"
        else:
            problem = "no usable verdict returned"
        verdicts.update(got)
        missing = batch[~batch["id"].astype(int).isin(got)]
        if missing.empty:
            return
        if len(batch) == 1:
            failures[int(missing.iloc[0]["id"])] = problem
            return
        middle = max(1, len(missing) // 2)
        for part in (missing.iloc[:middle], missing.iloc[middle:]):
            if not part.empty:
                run_batch(part)

    size = AGREEMENT_BATCH_SIZE if JUDGE_BACKEND == "groq" else 1
    batches = [events_df.iloc[start:start + size] for start in range(0, len(events_df), size)]
    reported = 0   # verdicts already summarised in a progress line
    for position, batch in enumerate(batches, start=1):
        run_batch(batch)
        if JUDGE_BACKEND == "groq" or position % 20 == 0 or position == len(batches):
            print(f"Judged batch {position}/{len(batches)} ({len(verdicts)} verdicts so far)", flush=True)
            all_scores = [verdict["score"] for verdict in verdicts.values()]
            print(score_summary(all_scores[reported:], AGREEMENT_THRESHOLD, all_scores), flush=True)
            reported = len(verdicts)
        if JUDGE_BACKEND == "groq" and position < len(batches):
            time.sleep(AGREEMENT_SLEEP_BETWEEN_CALLS)
    return verdicts, failures


to_judge = matched_events[matched_events["agreement_status"] == "judged"]
agreement_verdicts, agreement_failures = judge_agreement(to_judge)

print(f"\nEvents sent to the judge: {len(to_judge)}")
print(f"Verdicts returned:        {len(agreement_verdicts)}")
print(f"Not sent (no reference):  {int((matched_events['agreement_status'] == 'no_reference').sum())}")
if agreement_failures:
    print(f"Failed after retries:     {len(agreement_failures)} (left out of the table; rerun to retry)")


Judged batch 20/1788 (19 verdicts so far)
    scores: 1.0x1  0.9x1  0.7x15  0.1x1  0.0x1  | pass 17/19 (89%)  | cumulative 17/19 (89%)
Judged batch 40/1788 (39 verdicts so far)
    scores: 1.0x2  0.7x16  0.5x1  0.0x1  | pass 18/20 (90%)  | cumulative 35/39 (90%)
Judged batch 60/1788 (59 verdicts so far)
    scores: 1.0x3  0.7x17  | pass 20/20 (100%)  | cumulative 55/59 (93%)
Judged batch 80/1788 (79 verdicts so far)
    scores: 1.0x1  0.7x12  0.6x2  0.4x1  0.1x1  0.0x3  | pass 13/20 (65%)  | cumulative 68/79 (86%)
Judged batch 100/1788 (99 verdicts so far)
    scores: 1.0x1  0.7x18  0.1x1  | pass 19/20 (95%)  | cumulative 87/99 (88%)
Judged batch 120/1788 (119 verdicts so far)
    scores: 1.0x1  0.9x1  0.7x15  0.1x2  0.0x1  | pass 17/20 (85%)  | cumulative 104/119 (87%)
Judged batch 140/1788 (139 verdicts so far)
    scores: 0.7x13  0.4x1  0.1x5  0.0x1  | pass 13/20 (65%)  | cumulative 117/139 (84%)
Judged batch 160/1788 (159 verdicts so far)
    scores: 0.7x2  0.5x8  0.2x1  0.1x7  0.0

## 14. Build the agreement report

One row per generated event, in the shape of `bronze.annotations_report` so the QA notebook's analysis still
works: the matched-pair columns plus `llm_score`, `llm_match`, `llm_reason`, `eval_status`, `prompt_version`,
`model`, `evaluated_at`. Events the judge failed on are `judge_missing` and are not written.


In [29]:
agreement_report = matched_events.copy()
agreement_report["llm_score"] = agreement_report["id"].map(lambda i: agreement_verdicts.get(int(i), {}).get("score"))
agreement_report["llm_reason"] = agreement_report["id"].map(lambda i: agreement_verdicts.get(int(i), {}).get("reason"))
agreement_report["llm_match"] = agreement_report["llm_score"].map(
    lambda score: None if pd.isna(score) else bool(score >= AGREEMENT_THRESHOLD)
).astype("object")

agreement_report["eval_status"] = "scored"
no_reference = agreement_report["agreement_status"] == "no_reference"
agreement_report.loc[no_reference, "eval_status"] = "no_reference"
agreement_report.loc[no_reference, "llm_reason"] = "No MEVA reference to compare against."
missing_verdict = (agreement_report["eval_status"] == "scored") & agreement_report["llm_score"].isna()
agreement_report.loc[missing_verdict, "eval_status"] = "judge_missing"

agreement_report["prompt_version"] = AGREEMENT_PROMPT_VERSION
agreement_report["model"] = JUDGE_MODEL
agreement_report.loc[no_reference, "model"] = None
agreement_report["created_at"] = datetime.now(timezone.utc)
agreement_report["evaluated_at"] = datetime.now(timezone.utc)

print("Agreement rows:", len(agreement_report))
print(agreement_report["eval_status"].value_counts().to_string())
agreement_report[["video_name", "model_event_name", "meva_reference", "llm_score", "llm_match", "llm_reason", "eval_status"]].head(30)


Agreement rows: 1955
eval_status
scored           1785
no_reference      167
judge_missing       3


,video_name,model_event_name,meva_reference,llm_score,llm_match,llm_reason,eval_status
0,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,a vehicle reverses,0.0,False,vehicle reverses: opposite action (moving stra...,scored
1,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,a vehicle reverses,0.7,True,"vehicle reverses: partial evidence, no reversa...",scored
2,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,(no matching MEVA ground truth found),NaN,None,No MEVA reference to compare against.,no_reference
3,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV drives past the camera,(no matching MEVA ground truth found),NaN,None,No MEVA reference to compare against.,no_reference
4,2018-03-05.13-15-00.13-20-00.bus.G340,White SUV enters the frame and parks,a vehicle stops,0.7,True,"vehicle stops: parks implies stops, direct res...",scored
5,2018-03-05.13-15-00.13-20-00.bus.G340,Man walks toward a parked white SUV,a vehicle stops,0.7,True,"vehicle stops: the SUV remains stationary, no ...",scored
6,2018-03-05.13-15-00.13-20-00.bus.G340,Man opens the trunk of a white SUV and retriev...,a person opens a trunk; a person exits a vehic...,0.9,True,person opens a trunk; same action described (s...,scored
7,2018-03-05.13-15-00.13-20-00.bus.G340,Man pulls a suitcase from a white SUV,a person exits a vehicle; a person unloads a v...,0.7,True,"person closes a trunk: partial evidence, trunk...",scored
8,2018-03-05.13-15-00.13-20-00.bus.G340,Two men interact near a white SUV,a person talks to another person; a person clo...,NaN,None,NaN,judge_missing
9,2018-03-05.13-15-00.13-20-00.bus.G340,Two men interact near a white SUV while handli...,a person talks to another person; a person car...,0.7,True,person carries a heavy object; person picks up...,scored


## 15. Agreement metrics

Match rate and score distribution over judged events, then per video. A pile of scores between 0.2 and 0.6 means the
descriptions are consistent with MEVA's actors but describe position or movement rather than the MEVA action.


In [30]:
scored = agreement_report[agreement_report["eval_status"] == "scored"].copy()
scored["llm_match"] = scored["llm_match"].astype(bool)

print(f"Generated events in scope:        {len(agreement_report)}")
print(f"  judged against a MEVA reference: {len(scored)}")
print(f"  no MEVA reference in window:     {int((agreement_report['eval_status'] == 'no_reference').sum())}")
print(f"  judge failed:                    {int((agreement_report['eval_status'] == 'judge_missing').sum())}")
if len(scored):
    print(f"\nMean score (judged):              {scored['llm_score'].mean():.3f}")
    print(f"Agreement rate (score >= {AGREEMENT_THRESHOLD}): {scored['llm_match'].mean():.1%}")
    print("\nScore distribution:")
    print(scored["llm_score"].value_counts().reindex(list(reversed(ALLOWED_SCORES)), fill_value=0).to_string())

agreement_by_video = (
    scored.groupby(["camera_id", "video_name"])
    .agg(events=("id", "count"), matched=("llm_match", "sum"), mean_score=("llm_score", "mean"))
    .assign(agreement=lambda df: (df["matched"] / df["events"]).round(3), mean_score=lambda df: df["mean_score"].round(3))
    .sort_values("agreement")
)
print("\nAgreement by video (worst first):")
agreement_by_video


Generated events in scope:        1955
  judged against a MEVA reference: 1785
  no MEVA reference in window:     167
  judge failed:                    3

Mean score (judged):              0.554
Agreement rate (score >= 0.7): 63.2%

Score distribution:
llm_score
1.0      35
0.9       9
0.8       0
0.7    1085
0.6       9
0.5     282
0.4      17
0.3       0
0.2      39
0.1     250
0.0      59

Agreement by video (worst first):


events  matched  \
camera_id video_name                                                    
G301      2018-03-07.16-50-07.16-55-07.hospital.G301       1        0   
G508      2018-03-07.11-10-00.11-15-00.bus.G508           25        6   
G331      2018-03-07.16-50-00.16-55-00.bus.G331           71       20   
G339      2018-03-07.11-10-07.11-15-07.school.G339        46       13   
G436      2018-03-05.14-05-00.14-10-00.hospital.G436       7        2   
...                                                      ...      ...   
G419      2018-03-07.11-05-01.11-10-01.school.G419         2        2   
G420      2018-03-05.14-00-00.14-05-00.school.G420         1        1   
          2018-03-05.13-20-01.13-25-01.school.G420         3        3   
G424      2018-03-05.14-10-00.14-15-00.school.G424        10       10   
G420      2018-03-05.14-10-00.14-15-00.school.G420         2        2   

                                                      mean_score  agreement  
camera_id video_name                                                         
G301      2018-03-07.16-50-07.16-55-07.hospital.G301       0.500      0.000  
G508      2018-03-07.11-10-00.11-15-00.bus.G508            0.308      0.240  
G331      2018-03-07.16-50-00.16-55-00.bus.G331            0.408      0.282  
G339      2018-03-07.11-10-07.11-15-07.school.G339         0.341      0.283  
G436      2018-03-05.14-05-00.14-10-00.hospital.G436       0.357      0.286  
...                                                          ...        ...  
G419      2018-03-07.11-05-01.11-10-01.school.G419         0.700      1.000  
G420      2018-03-05.14-00-00.14-05-00.school.G420         0.700      1.000  
          2018-03-05.13-20-01.13-25-01.school.G420         0.900      1.000  
G424      2018-03-05.14-10-00.14-15-00.school.G424         0.850      1.000  
G420      2018-03-05.14-10-00.14-15-00.school.G420         0.800      1.000  

[70 rows x 4 columns]

In [31]:
# Events the judge scored lowest: what the pipeline described versus what MEVA says happened then.
scored.sort_values("llm_score")[
    ["video_name", "event_start_frame", "event_end_frame", "model_event_name", "model_description",
     "meva_reference", "llm_score", "llm_reason"]
].head(25)


,video_name,event_start_frame,event_end_frame,model_event_name,model_description,meva_reference,llm_score,llm_reason
0,2018-03-05.13-15-00.13-20-00.bus.G340,939,1020,White SUV drives past the camera,A white SUV enters the frame from the right si...,a vehicle reverses,0.0,vehicle reverses: opposite action (moving stra...
1004,2018-03-07.11-05-01.11-10-01.school.G328,5845,6060,Pink car drives past parked vehicles,A pink car is seen driving from the right side...,a person talks to another person,0.0,"person talks: unrelated, no matching activity ..."
387,2018-03-07.11-00-00.11-05-00.school.G420,467,540,Two people walk down the stairs,Two people walk down a staircase from the uppe...,a person talks to another person,0.0,person talks to another person: opposite actio...
995,2018-03-07.11-05-01.11-10-01.school.G328,3631,3900,Person walks across the grass,A person walks across the grass from the right...,a person talks to another person,0.0,person talks: opposite action (walks vs talks)...
994,2018-03-07.11-05-01.11-10-01.school.G328,3360,3628,Silver sedan drives past the building,A silver sedan enters the frame from the botto...,a person talks to another person,0.0,person talks: unrelated activity (step 1)
984,2018-03-07.11-05-01.11-10-01.school.G328,1440,1740,A black pickup truck drives past parked vehicles,"A black pickup truck, identified as truck, is ...",a person talks to another person; a person ope...,0.0,vehicle stops: opposite action (moving vs park...
95,2018-03-05.13-15-00.13-20-00.school.G424,6960,7260,Person walks across a grassy area,A person wearing dark clothing walks steadily ...,a person carries a heavy object,0.0,person carries a heavy object: opposite action...
1705,2018-03-07.16-50-00.16-55-00.bus.G340,8644,8836,Person walks toward a parked car,A person wearing a dark jacket walks from the ...,a person talks to another person,0.0,person talks: opposite action (walks vs talks)...
521,2018-03-07.11-00-01.11-05-01.bus.G506,7920,8205,Man in red shirt walks toward the building ent...,A man wearing a red long-sleeved shirt and blu...,a person talks to another person; a person tex...,0.0,person talks to another person; no matching ac...
523,2018-03-07.11-00-01.11-05-01.bus.G506,8400,8596,Man in red jacket walks away from the building,A man wearing a red jacket and blue jeans stan...,a person talks to another person; a person tex...,0.0,"person talks: opposite action (no interaction,..."


## 16. Write to `bronze.annotations_report`

The table keeps every prompt version, so earlier judges stay comparable. This run replaces the rows of
`AGREEMENT_PROMPT_VERSION` for the events in scope (all events, or the filtered videos' events) in one
transaction, and leaves other prompt versions untouched. `judge_missing` rows are not written. The table is
created with the QA notebook's columns when it does not exist yet.


In [32]:
AGREEMENT_COLUMNS = [
    "id", "event_id", "camera_id", "model_event_name", "model_description", "event_start_frame",
    "event_end_frame", "meva_reference", "n_meva_activities_matched", "created_at", "has_ground_truth",
    "llm_score", "llm_match", "llm_reason", "eval_status", "prompt_version", "model", "evaluated_at",
]
AGREEMENT_DDL = f"""
CREATE TABLE IF NOT EXISTS bronze.{AGREEMENT_TABLE} (
    id BIGINT,
    event_id TEXT,
    camera_id TEXT,
    model_event_name TEXT,
    model_description TEXT,
    event_start_frame INTEGER,
    event_end_frame INTEGER,
    meva_reference TEXT,
    n_meva_activities_matched INTEGER,
    created_at TIMESTAMPTZ,
    has_ground_truth BOOLEAN
);
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_score DOUBLE PRECISION;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_match BOOLEAN;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_reason TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS eval_status TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS prompt_version TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS model TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS evaluated_at TIMESTAMPTZ;
CREATE UNIQUE INDEX IF NOT EXISTS {AGREEMENT_TABLE}_id_prompt_version_uq ON bronze.{AGREEMENT_TABLE} (id, prompt_version);
"""

agreement_writable = agreement_report[agreement_report["eval_status"] != "judge_missing"][AGREEMENT_COLUMNS].copy()
agreement_writable["llm_match"] = agreement_writable["llm_match"].where(agreement_writable["llm_match"].notna(), None)
agreement_writable["llm_score"] = agreement_writable["llm_score"].astype("float64")

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(agreement_writable)} rows kept local, nothing written.")
elif agreement_writable.empty:
    print("No agreement rows with a verdict - the stored table is left untouched.")
else:
    with engine.begin() as conn:
        for statement in AGREEMENT_DDL.strip().split(";"):
            if statement.strip():
                conn.execute(text(statement))
        if VIDEO_FILTER:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{AGREEMENT_TABLE} WHERE prompt_version = :pv AND event_id = ANY(:ids);"),
                {"pv": AGREEMENT_PROMPT_VERSION, "ids": list(matched_events["event_id"])},
            ).rowcount
        else:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{AGREEMENT_TABLE} WHERE prompt_version = :pv;"),
                {"pv": AGREEMENT_PROMPT_VERSION},
            ).rowcount
        agreement_writable.to_sql(
            AGREEMENT_TABLE, conn, schema="bronze", if_exists="append", index=False, method=psql_insert_copy,
        )
    print(f"bronze.{AGREEMENT_TABLE} ({AGREEMENT_PROMPT_VERSION}): {deleted} old rows removed, {len(agreement_writable)} rows written.")

held_back = int((agreement_report["eval_status"] == "judge_missing").sum())
if held_back:
    print(f"Held back {held_back} event(s) without a verdict; re-run the notebook to retry them.")


bronze.annotations_report (v4_unified_scale_judge): 0 old rows removed, 1952 rows written.
Held back 3 event(s) without a verdict; re-run the notebook to retry them.


In [33]:
if WRITE_TO_DB:
    print(f"bronze.{AGREEMENT_TABLE} contents by prompt version:")
    display(pd.read_sql(
        text(f"""
        SELECT prompt_version, eval_status, COUNT(*) AS rows,
               ROUND(AVG(llm_score)::numeric, 3) AS avg_llm_score,
               ROUND(AVG(CASE WHEN llm_match THEN 1.0 ELSE 0.0 END)::numeric, 3) AS match_rate
        FROM bronze.{AGREEMENT_TABLE}
        GROUP BY prompt_version, eval_status
        ORDER BY prompt_version, eval_status;
        """),
        engine,
    ))


bronze.annotations_report contents by prompt version:


,prompt_version,eval_status,rows,avg_llm_score,match_rate
0,v2.3_semantic_judge,no_reference,183,NaN,0.000
1,v2.3_semantic_judge,scored,739,0.557,0.153
2,v2.4_semantic_judge,no_reference,167,NaN,0.000
3,v2.4_semantic_judge,scored,1788,0.583,0.197
4,v3_merged_semantic_judge,no_reference,167,NaN,0.000
5,v3_merged_semantic_judge,scored,1788,0.424,0.155
6,v4_unified_scale_judge,no_reference,167,NaN,0.000
7,v4_unified_scale_judge,scored,1785,0.554,0.632


# Both directions together

## 17. Coverage and agreement side by side

Coverage (Part A) is the share of MEVA activity types the pipeline described at least once per video. Agreement
(Part B) is the share of generated events whose text matches a MEVA activity in their own time window. Together
they read like recall and precision: a pipeline can score well on one by sacrificing the other, so report both.


In [34]:
coverage_pairs = len(evaluable)
coverage_rate = evaluable["covered"].mean() if coverage_pairs else float("nan")
agreement_events = len(scored)
agreement_rate = scored["llm_match"].mean() if agreement_events else float("nan")

summary_both = pd.DataFrame([
    {"measure": f"Coverage (MEVA activity types described, score >= {COVERAGE_THRESHOLD})", "n": coverage_pairs,
     "rate": round(float(coverage_rate), 3) if coverage_pairs else None, "table": f"bronze.{COVERAGE_TABLE}"},
    {"measure": f"Agreement (generated events matching MEVA, score >= {AGREEMENT_THRESHOLD})", "n": agreement_events,
     "rate": round(float(agreement_rate), 3) if agreement_events else None, "table": f"bronze.{AGREEMENT_TABLE}"},
])
print(f"Judge: {JUDGE_MODEL} ({JUDGE_BACKEND})" + (f" · videos matching {VIDEO_FILTER!r}" if VIDEO_FILTER else " · all videos"))
display(summary_both)

per_video_coverage = (
    evaluable.groupby("video_name")
    .agg(activity_types=("pair_id", "count"), covered=("covered", "sum"))
    .assign(coverage=lambda df: (df["covered"] / df["activity_types"]).round(3))
)
per_video_agreement = (
    scored.groupby("video_name")
    .agg(events_judged=("id", "count"), matched=("llm_match", "sum"))
    .assign(agreement=lambda df: (df["matched"] / df["events_judged"]).round(3))
)
side_by_side = per_video_coverage.join(per_video_agreement, how="outer").sort_values("coverage")
print("\nPer video (worst coverage first):")
side_by_side


Judge: mlx-community/Qwen3-8B-4bit (mlx) · all videos


,measure,n,rate,table
0,"Coverage (MEVA activity types described, score...",605,0.805,bronze.meva_event_coverage
1,"Agreement (generated events matching MEVA, sco...",1785,0.632,bronze.annotations_report



Per video (worst coverage first):


,activity_types,covered,coverage,events_judged,matched,agreement
video_name,,,,,,
2018-03-07.16-50-07.16-55-07.hospital.G301,2,0,0.000,1,0,0.000
2018-03-05.13-15-00.13-20-00.bus.G505,1,0,0.000,9,9,1.000
2018-03-05.14-05-00.14-09-59.bus.G506,2,0,0.000,1,1,1.000
2018-03-07.11-10-01.11-15-01.bus.G505,3,0,0.000,15,14,0.933
2018-03-07.16-50-01.16-55-01.bus.G505,3,1,0.333,17,17,1.000
...,...,...,...,...,...,...
2018-03-07.11-05-01.11-10-01.school.G328,15,15,1.000,41,29,0.707
2018-03-05.13-20-01.13-25-01.school.G419,3,3,1.000,3,3,1.000
2018-03-05.13-20-01.13-25-01.school.G420,3,3,1.000,3,3,1.000


## 18. Results from the stored tables

These queries read the tables written above (`meva_event_coverage`, `matched_pairs`, `annotations_report`) straight
from the database, so they show what is actually stored. They cover only videos that have both pipeline events and
MEVA ground truth (and match `VIDEO_FILTER` when it is set). Scores are recomputed from the stored `coverage_score`
and `llm_score` against `COVERAGE_THRESHOLD` and `AGREEMENT_THRESHOLD`; agreement uses `AGREEMENT_PROMPT_VERSION`.

The first cell defines the queries; each result below has its own cell.

In [35]:
RESULTS_SCOPE_CTE = """
scope AS (
    SELECT v.video_id, v.video_name
    FROM bronze.videos v
    WHERE EXISTS (SELECT 1 FROM bronze.events e WHERE e.video_id = v.video_id)
      AND EXISTS (SELECT 1 FROM bronze.ground_truth g
                  WHERE g.video_id = v.video_id AND NOT (g.activity_type = ANY(:ignored)))
      AND (CAST(:pattern AS text) IS NULL OR v.video_name LIKE :pattern)
)"""

RESULTS_COVERAGE_SQL = f"""
WITH {RESULTS_SCOPE_CTE}
SELECT
    COUNT(*)                                                         AS activity_type_pairs,
    COUNT(*) FILTER (WHERE c.coverage_score >= :cov_t)               AS covered_pairs,
    ROUND(AVG(CASE WHEN c.coverage_score >= :cov_t THEN 1.0 ELSE 0.0 END), 3) AS coverage_rate,
    SUM(c.n_meva_instances)                                          AS meva_instances,
    COALESCE(SUM(c.n_meva_instances) FILTER (WHERE c.coverage_score >= :cov_t), 0) AS instances_in_covered_types
FROM bronze.meva_event_coverage c
JOIN scope USING (video_id)
WHERE c.status = 'judged';
"""

# Every pipeline event in scope: does its window overlap any MEVA activity in time, how closely (best temporal
# IoU with a single MEVA instance), and did the agreement judge pass it?
RESULTS_EVENT_SQL = f"""
WITH {RESULTS_SCOPE_CTE},
ev AS (
    SELECT mp.event_id, mp.n_meva_activities_matched > 0 AS time_matched,
           best.iou AS best_iou, ar.eval_status, ar.llm_score
    FROM bronze.matched_pairs mp
    JOIN scope USING (video_id)
    JOIN bronze.events e ON e.event_id = mp.event_id
    LEFT JOIN LATERAL (
        SELECT MAX((LEAST(g.end_frame, e.end_frame) - GREATEST(g.start_frame, e.start_frame) + 1)::numeric
                   / (GREATEST(g.end_frame, e.end_frame) - LEAST(g.start_frame, e.start_frame) + 1)) AS iou
        FROM bronze.ground_truth g
        WHERE g.video_id = e.video_id AND NOT (g.activity_type = ANY(:ignored))
          AND g.start_frame <= e.end_frame AND g.end_frame >= e.start_frame
    ) best ON TRUE
    LEFT JOIN bronze.annotations_report ar ON ar.event_id = mp.event_id AND ar.prompt_version = :pv
)
SELECT
    COUNT(*)                                                     AS events,
    COUNT(*) FILTER (WHERE NOT time_matched)                     AS no_time_match,
    COUNT(*) FILTER (WHERE time_matched)                         AS time_matched,
    ROUND(AVG(best_iou) FILTER (WHERE time_matched), 3)          AS avg_best_iou,
    COUNT(*) FILTER (WHERE time_matched AND eval_status = 'scored') AS time_matched_judged,
    COUNT(*) FILTER (WHERE time_matched AND llm_score >= :agr_t) AS time_matched_passed
FROM ev;
"""

# One row per MEVA instance in scope with the four coverage flags described above.
RESULTS_INSTANCE_FLAGS = f"""
WITH {RESULTS_SCOPE_CTE},
gt AS (
    SELECT g.id, g.video_id, scope.video_name, g.activity_type, g.start_frame, g.end_frame
    FROM bronze.ground_truth g
    JOIN scope USING (video_id)
    WHERE NOT (g.activity_type = ANY(:ignored))
),
flags AS (
    SELECT
        gt.id, gt.video_name,
        EXISTS (
            SELECT 1 FROM bronze.events e
            WHERE e.video_id = gt.video_id AND e.start_frame <= gt.end_frame AND e.end_frame >= gt.start_frame
        ) AS time_covered,
        EXISTS (
            SELECT 1 FROM bronze.events e
            JOIN bronze.annotations_report ar ON ar.event_id = e.event_id AND ar.prompt_version = :pv
            WHERE e.video_id = gt.video_id AND e.start_frame <= gt.end_frame AND e.end_frame >= gt.start_frame
              AND ar.llm_score >= :agr_t
        ) AS time_and_agreement,
        COALESCE(c.coverage_score >= :cov_t, FALSE) AS activity_covered,
        EXISTS (
            SELECT 1 FROM bronze.events e
            WHERE e.event_id = c.covering_event_id AND c.coverage_score >= :cov_t
              AND e.start_frame <= gt.end_frame AND e.end_frame >= gt.start_frame
        ) AS activity_and_time
    FROM gt
    LEFT JOIN bronze.meva_event_coverage c
           ON c.video_id = gt.video_id AND c.activity_type = gt.activity_type AND c.status = 'judged'
)"""

RESULTS_INSTANCE_SQL = RESULTS_INSTANCE_FLAGS + """
SELECT
    COUNT(*)                                    AS meva_instances,
    COUNT(*) FILTER (WHERE time_covered)        AS time_covered,
    COUNT(*) FILTER (WHERE time_and_agreement)  AS time_and_agreement,
    COUNT(*) FILTER (WHERE activity_covered)    AS activity_covered,
    COUNT(*) FILTER (WHERE activity_and_time)   AS activity_and_time
FROM flags;
"""

RESULTS_PER_VIDEO_SQL = RESULTS_INSTANCE_FLAGS + """
SELECT
    video_name,
    COUNT(*)                                                                  AS meva_instances,
    ROUND(AVG(CASE WHEN time_covered       THEN 1.0 ELSE 0.0 END), 3)        AS time_covered,
    ROUND(AVG(CASE WHEN time_and_agreement THEN 1.0 ELSE 0.0 END), 3)        AS time_and_agreement,
    ROUND(AVG(CASE WHEN activity_covered   THEN 1.0 ELSE 0.0 END), 3)        AS activity_covered,
    ROUND(AVG(CASE WHEN activity_and_time  THEN 1.0 ELSE 0.0 END), 3)        AS activity_and_time
FROM flags
GROUP BY video_name
ORDER BY activity_and_time, video_name;
"""

results_params = {
    "ignored": list(IGNORED_MEVA_TYPES),
    "pattern": f"%{VIDEO_FILTER}%" if VIDEO_FILTER else None,
    "pv": AGREEMENT_PROMPT_VERSION,
    "cov_t": COVERAGE_THRESHOLD,
    "agr_t": AGREEMENT_THRESHOLD,
}


def funnel(stages):
    """(label, count, base) rows -> a table with the share of the base and of the first row."""
    total = stages[0][1] or 0
    return pd.DataFrame([
        {"stage": label, "n": int(n or 0),
         "of": base_label, "share": round(int(n or 0) / base, 3) if base else None,
         "share_of_total": round(int(n or 0) / total, 3) if total else None}
        for label, n, base_label, base in stages
    ])


print(f"Prompt version {AGREEMENT_PROMPT_VERSION} · coverage pass >= {COVERAGE_THRESHOLD} · agreement pass >= {AGREEMENT_THRESHOLD}"
      + (f" · videos matching {VIDEO_FILTER!r}" if VIDEO_FILTER else " · all videos with events and ground truth"))

Prompt version v4_unified_scale_judge · coverage pass >= 0.7 · agreement pass >= 0.7 · all videos with events and ground truth


### 18.1 Coverage

The share of (video, MEVA activity type) pairs whose coverage score reaches the threshold, and how many MEVA
instances those covered activity types represent.

In [36]:
cov = pd.read_sql(text(RESULTS_COVERAGE_SQL), engine, params=results_params).iloc[0]

funnel([
    ("activity type pairs", cov["activity_type_pairs"], None, None),
    (f"covered (score >= {COVERAGE_THRESHOLD})", cov["covered_pairs"], "activity type pairs", cov["activity_type_pairs"]),
    ("MEVA instances in those pairs", cov["meva_instances"], None, None),
    ("instances whose type is covered", cov["instances_in_covered_types"], "MEVA instances", cov["meva_instances"]),
]).drop(columns="share_of_total")

,stage,n,of,share
0,activity type pairs,605,NaN,NaN
1,covered (score >= 0.7),487,activity type pairs,0.805
2,MEVA instances in those pairs,2720,NaN,NaN
3,instances whose type is covered,2337,MEVA instances,0.859


### 18.2 Time match of the pipeline event windows

Time only, no judge: how many event windows overlap no MEVA activity at all, how many overlap at least one, and
how closely the overlapping ones line up (the mean of each event's best temporal IoU with a single MEVA instance).

In [37]:
ev = pd.read_sql(text(RESULTS_EVENT_SQL), engine, params=results_params).iloc[0]

print(f"Mean best temporal IoU of time-matched events with a single MEVA instance: {ev['avg_best_iou']}")
funnel([
    ("pipeline events", ev["events"], None, None),
    ("no MEVA activity in the window (time miss)", ev["no_time_match"], "pipeline events", ev["events"]),
    ("overlap a MEVA activity in time", ev["time_matched"], "pipeline events", ev["events"]),
])

Mean best temporal IoU of time-matched events with a single MEVA instance: 0.404


,stage,n,of,share,share_of_total
0,pipeline events,1954,NaN,NaN,1.000
1,no MEVA activity in the window (time miss),166,pipeline events,0.085,0.085
2,overlap a MEVA activity in time,1788,pipeline events,0.915,0.915


### 18.3 Agreement of the time-matched events

Of the events whose window overlaps a MEVA activity (18.2), how many the agreement judge scored, and how many
reach `AGREEMENT_THRESHOLD`. Uses `ev` from 18.2.

In [38]:
if ev["time_matched"] and not ev["time_matched_judged"]:
    print(f"WARNING: no {AGREEMENT_PROMPT_VERSION} verdicts stored yet - run Part B with WRITE_TO_DB = True first.")

funnel([
    ("time-matched events", ev["time_matched"], None, None),
    ("judged", ev["time_matched_judged"], "time matched", ev["time_matched"]),
    (f"pass agreement (score >= {AGREEMENT_THRESHOLD})", ev["time_matched_passed"], "time matched", ev["time_matched"]),
]).assign(share_of_all_events=lambda df: (df["n"] / ev["events"]).round(3) if ev["events"] else None)

,stage,n,of,share,share_of_total,share_of_all_events
0,time-matched events,1788,NaN,NaN,1.000,0.915
1,judged,1785,time matched,0.998,0.998,0.914
2,pass agreement (score >= 0.7),1129,time matched,0.631,0.631,0.578


### 18.4 MEVA instances covered by time and activity

Every ground-truth instance in these videos, from loosest to strictest:

* `time_covered`: some pipeline event overlaps it in time.
* `time_and_agreement`: an overlapping event passed the agreement judge (its text fits a MEVA activity in its
  window, not necessarily this one).
* `activity_covered`: its activity type is covered somewhere in the video (Part A, time ignored).
* `activity_and_time`: its activity type is covered **and** the event credited with covering it overlaps this
  instance in time. This is the strictest count: Part A credits one event per activity type, so other instances
  of the same type at other times only count if that event spans them too.

In [39]:
inst = pd.read_sql(text(RESULTS_INSTANCE_SQL), engine, params=results_params).iloc[0]

funnel([
    ("MEVA instances", inst["meva_instances"], None, None),
    ("time covered (an event overlaps it)", inst["time_covered"], "MEVA instances", inst["meva_instances"]),
    ("time + an overlapping event passes agreement", inst["time_and_agreement"], "MEVA instances", inst["meva_instances"]),
    ("activity type covered (time ignored)", inst["activity_covered"], "MEVA instances", inst["meva_instances"]),
    ("activity covered + covering event overlaps it", inst["activity_and_time"], "MEVA instances", inst["meva_instances"]),
]).drop(columns="share_of_total")

,stage,n,of,share
0,MEVA instances,2720,NaN,NaN
1,time covered (an event overlaps it),2502,MEVA instances,0.920
2,time + an overlapping event passes agreement,2239,MEVA instances,0.823
3,activity type covered (time ignored),2337,MEVA instances,0.859
4,activity covered + covering event overlaps it,255,MEVA instances,0.094


### 18.5 Per video

The four instance shares from 18.4 for each video, worst `activity_and_time` first.

In [40]:
pd.read_sql(text(RESULTS_PER_VIDEO_SQL), engine, params=results_params)

,video_name,meva_instances,time_covered,time_and_agreement,activity_covered,activity_and_time
0,2018-03-05.13-15-00.13-20-00.bus.G505,4,1.000,1.000,0.000,0.00
1,2018-03-05.13-15-01.13-20-01.school.G328,19,0.211,0.211,0.368,0.00
2,2018-03-05.13-20-00.13-25-00.bus.G340,9,0.667,0.667,1.000,0.00
3,2018-03-05.14-05-00.14-09-59.bus.G506,2,0.500,0.500,0.000,0.00
4,2018-03-05.14-10-00.14-15-00.school.G424,17,0.706,0.706,0.412,0.00
...,...,...,...,...,...,...
65,2018-03-05.13-20-01.13-25-01.school.G420,4,0.750,0.750,1.000,0.75
66,2018-03-05.13-15-00.13-20-00.school.G423,2,1.000,1.000,1.000,1.00
67,2018-03-05.14-00-00.14-05-00.school.G420,2,1.000,1.000,1.000,1.00
68,2018-03-07.11-05-01.11-10-01.school.G419,3,1.000,1.000,1.000,1.00
